# Cascade Precision Products — 13-Week Cash Flow Agent

**A complete agent harness for treasury forecasting, in one notebook.** This file is
self-contained: run it top to bottom in Google Colab and it builds the harness, loads a
frozen synthetic dataset, runs the deterministic and probabilistic engines, drives a
Claude-powered agent through the full pipeline with a *real* human review gate, and hands
you the sponsor deliverable — a traditional 13-week cash flow workbook. The only thing you
supply is an `ANTHROPIC_API_KEY`.

## The story

Cascade Precision Products ($72M contract manufacturer of precision-machined aerospace,
defense, and industrial components; ~240 employees) was acquired ten weeks ago by
**Granite Peak Partners** in a leveraged buyout. The sponsor's first standing ask: a
weekly 13-week cash flow forecast with covenant visibility. Week one under new ownership —
prove you can see cash.

## The three layers this build teaches

| Layer | What it looks like here |
|---|---|
| **Governance** | A \$1.5M operating cash floor, a \$4.0M liquidity covenant tested at week 13, a human sign-off gate that actually blocks the pipeline, and a complete audit trail |
| **Finance** | Where is the trough, what threatens the covenant, and what should management do about it |
| **Capability** | A deterministic roll-forward engine, plus a Monte Carlo layer that turns a single-point forecast into a distribution |

## The punchline

**The single-point forecast passes the covenant with \$631K to spare. The distribution
says there is a ~20% chance it doesn't.** Same data, same engine — one more question asked
of it. Deterministic control, probabilistic reasoning. That is the whole demo.

## A note for the FP&A reader: yes, this is an unusual way to build a TWCF

A traditional 13-week cash flow is a single spreadsheet: one number per week, receipts
penciled on due dates or the analyst's haircut, and a bottom line that either clears the
covenant or doesn't. This build departs from that in two ways, and both departures have
a serious claim to being the *better* practice:

**Departure 1 — collections are forecast from observed payment behavior, not stated
terms.** Every customer here carries its measured days-to-pay and the volatility around
it, and the forecast collects on behavior. Good treasury teams already do a version of
this (DSO-based timing, roll rates); this build just makes it explicit, per customer,
and auditable. Vantage Aerospace is on Net 45 and pays in 64 days — penciling its \$2.1M
milestone on the due date isn't a forecast, it's a hope.

**Departure 2 — the forecast is a distribution, not a line.** This is the unusual part,
and the case for it:

1. **Headroom is not probability, and the point estimate hides the difference.** Two
   companies can both show \$631K of week-13 headroom — one with sleepy, stable
   receivables, one with a fifth of its AR concentrated in a customer whose timing has
   started to wander. The classic TWCF cannot tell those companies apart. The
   distribution can: here, that same \$631K carries a ~20% breach probability.
2. **Treasury decisions are bets, and you can't price a bet without odds.** Should you
   offer Vantage an early-pay incentive? Pre-clear a covenant waiver with the sponsor?
   Defer the machine deposit? Each mitigation has a cost, and comparing cost against
   "headroom feels thin" is vibes. Comparing it against "cuts breach probability from
   20% to 1.3%" is analysis.
3. **You already do this — just informally.** Every base/upside/downside scenario deck
   is three hand-picked draws from a distribution you carry in your head. Monte Carlo is
   the same idea done honestly: a thousand draws, sampled from each customer's *measured*
   payment variance instead of from judgment, with the correlations that actually break
   covenants (a slow payer is slow on all its invoices at once).
4. **It replaces sandbagging with auditable assumptions.** The classic way to express
   risk in a TWCF is silent conservatism — pencil the big collection a week late "to be
   safe." That distorts the base case, compounds invisibly across line items, and can't
   be audited. Here the base case stays honest (expected behavior) and the risk lives
   explicitly in the sigmas, which are measured from history and can be challenged line
   by line.
5. **The only reason TWCFs are single-point is that Excel made distributions expensive.**
   Banks and insurers have run their liquidity this way for decades (ALM, VaR). Once the
   roll-forward is a pure function instead of a spreadsheet, a thousand replays cost a
   fraction of a second. The marginal price of the honest answer has gone to zero.

And the part that should reassure rather than alarm: **the deterministic forecast is not
replaced.** It is still built, still reviewed, still the workbook that goes to the
sponsor, and it still ties out to the dollar. The probabilistic layer is one additional
question asked of the same engine. That is what "deterministic control, probabilistic
reasoning" means in practice.

## How this notebook is organized

- **Part 0** — setup (dependencies, Google Drive, and the API key)
- **Part 1** — the harness, module by module, then the frozen dataset. Each cell writes
  one file of the `cashflow_harness` package (or one data file) with an explanation of
  what it does and why. This is the same package a FastAPI + React demo UI wraps in
  Phase 2; the notebook and the UI run identical logic and cannot drift.
- **Part 2** — run it: verify the dataset, run the engines and scenarios, then the agent
  with the review gate, the audit trail, and the workbook.

> If you edit a module cell after running Part 2, restart the runtime
> (Runtime → Restart session) and re-run — Python caches imports.

---
# Part 0 · Setup

Two dependencies Colab doesn't ship with (`anthropic`, `python-dotenv`), an optional
Google Drive mount, and the API key.

**Where does the data come from?** It ships inside this notebook. Section 1.4 embeds
the frozen synthetic dataset — the exact files, byte for byte, that the demo was
calibrated and tested on (anchored to the 2026-08-21 Friday close; week 1 opens Monday
Aug 24). Nothing is generated at run time, nothing is uploaded, and nothing is read from
your machine. Part 2.1 verifies the dataset against its control totals before anything
else runs.

**Where does everything live?** Your choice, next cell. By default the whole project
(package, data, outputs) is created in a folder in your **Google Drive**, so the
generated data, the audit trail, and the 13-week workbook persist across Colab sessions
and are browsable in the Drive UI. Set `USE_DRIVE = False` to work in the runtime's
ephemeral disk instead (everything still runs; it just vanishes when the runtime
recycles).

The API key resolves in this order: environment variable → `.env` file → Colab secret
(the key icon in the left sidebar, name it `ANTHROPIC_API_KEY`) → an interactive prompt.
It is never hardcoded anywhere in this project — that is checklist item one for any
finance artifact that leaves your laptop.

In [ ]:
%pip install -q anthropic python-dotenv openpyxl

In [ ]:
# Choose where the project lives. USE_DRIVE = True mounts Google Drive and
# works from MyDrive/cashflow-cfo-agent so data and outputs persist across
# sessions. False = the runtime's ephemeral disk.
USE_DRIVE = True

import os
import pathlib

if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        workdir = "/content/drive/MyDrive/cashflow-cfo-agent"
        os.makedirs(workdir, exist_ok=True)
        os.chdir(workdir)
    except ImportError:
        print("Not running in Colab — staying in the current directory.")

os.makedirs("cashflow_harness", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)
print("Project home:", pathlib.Path.cwd())

In [ ]:
# API key: env var -> .env -> Colab secret -> prompt. Never hardcoded.
if not os.getenv("ANTHROPIC_API_KEY"):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
if not os.getenv("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
print("API key loaded:", "yes" if os.getenv("ANTHROPIC_API_KEY") else "NO — the agent run in Part 2 will fail")

---
# Part 1 · The harness, module by module

Nine files. Read them in order and you have read the whole system. The design rule that
matters most: **every function is pure and importable.** The notebook calls them here;
the Phase 2 demo server imports the same functions unchanged. If a number ever differs
between the notebook and the UI, the wrapper is wrong, not the engine.

## 1.1 · `config.py` — the governance rails and the agent's charter

Everything the credit agreement and treasury policy fix in advance lives here as named
constants: the \$1.5M operating floor, the \$4.0M covenant and its week-13 test, the
\$250K draw increment, the sweep buffer, the Monte Carlo seed (reproducibility is a
governance feature — the same forecast twice gives the same answer).

It also holds the **system prompt**: the agent's role (treasury/FP&A analyst at Cascade,
newly LBO'd), the ten-tool workflow it must follow, and the writing rules for the
commentary (cite the driver behind every number, distinguish timing from structural,
no speculation beyond the data). The harness is model-agnostic — `MODEL` is one string,
swappable to Opus or a future model without touching anything else.

In [ ]:
%%writefile cashflow_harness/config.py
"""Configuration for the Cascade Precision 13-week cash flow harness.

The harness is model-agnostic. MODEL can be swapped to Opus or a future
model without touching the engine, tools, or agent loop.
"""

import os
from pathlib import Path

try:
    from dotenv import load_dotenv

    load_dotenv(Path(__file__).parent.parent / ".env", override=True)
except ImportError:
    # Colab or minimal installs supply ANTHROPIC_API_KEY via the environment
    # (e.g. google.colab userdata secrets) rather than a .env file.
    pass

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

MODEL = "claude-sonnet-5"  # swappable: any Claude model with tool use
MAX_TOKENS = 8192

DATA_DIR = Path(__file__).parent.parent / "data"
OUTPUT_DIR = Path(__file__).parent.parent / "output"

# Forecast horizon
NUM_WEEKS = 13

# Capital structure and treasury policy
BEGINNING_CASH = 2_400_000
REVOLVER_LIMIT = 15_000_000
REVOLVER_DRAWN = 6_000_000
OPERATING_FLOOR = 1_500_000     # auto-draw the revolver below this
SWEEP_BUFFER = 500_000          # sweep excess to the revolver above floor + buffer
DRAW_INCREMENT = 250_000        # revolver draws round up to this increment
COVENANT_THRESHOLD = 4_000_000  # cash + undrawn availability, tested at week 13
COVENANT_TEST_WEEK = 13

# Term loan quarterly debt service, due end of week 13
TERM_LOAN_BALANCE = 38_000_000
DEBT_SERVICE_AMORT = 1_250_000
DEBT_SERVICE_INTEREST = 950_000

# Monte Carlo
MC_ITERATIONS = 1_000
MC_SEED = 42

# Synthetic data generation. The dataset is FROZEN: a fixed seed and a fixed
# as-of anchor (the Friday close before the Aug 26 demo airing; week 1 opens
# Monday Aug 24). Regenerating always reproduces the committed files exactly.
GEN_SEED = 7
DATA_ASOF = "2026-08-21"


SYSTEM_PROMPT = """You are a treasury and FP&A analyst at Cascade Precision Products, \
a contract manufacturer of precision-machined components for aerospace, defense, and \
industrial OEMs, with roughly $72 million in trailing revenue and 240 employees. The \
company was acquired ten weeks ago by Granite Peak Partners in a leveraged buyout. \
The sponsor's first standing ask is a weekly 13-week cash flow forecast. This is week \
one under new ownership. Your job is to prove the company can see its cash.

Your work sits on three layers. Governance: the $1,500,000 operating cash floor, the \
$4,000,000 liquidity covenant tested at week 13, mandatory human sign-off before \
anything goes to the sponsor, and a complete audit trail. Finance: where the cash \
trough falls, what threatens the covenant, and what management should do about it. \
Capability: a deterministic roll-forward engine and a Monte Carlo layer that turns a \
single-point forecast into a distribution.

WORKFLOW

You have access to the following tools. Use them in this sequence:

1. load_facility_and_position -- Load beginning cash, the revolver, the operating \
floor, the covenant terms, and the debt service schedule.

2. load_receivables -- Load open AR with customer subtotals, concentration, and each \
customer's payment behavior. Pay attention to any customer whose recent payment \
timing has stretched beyond terms.

3. load_payables_and_fixed -- Load scheduled disbursements by week: open AP by due \
date plus the fixed schedule of payroll, rent, and one-time items. Note which items \
are discretionary.

4. build_deterministic_forecast -- Run the 13-week roll-forward with revolver \
mechanics. Identify the trough week and the week-13 covenant headroom.

5. run_monte_carlo -- Run the probabilistic layer over collection timing and sales \
variability. Report the P10/P50/P90 bands and the two breach probabilities. Compare \
what the distribution says against what the single-point forecast said.

6. run_scenario -- Test the what-ifs that matter for the covenant. At minimum run a \
driver sweep to rank what moves week-13 headroom most, then test the specific actions \
you are weighing (accelerating a collection, deferring a discretionary item).

7. compute_variance -- Bridge the trailing 4 weeks of actuals against the prior \
forecast. Attribute the miss to timing versus volume. This calibrates how much to \
trust the current forecast's timing assumptions.

8. draft_cash_narrative -- Write the treasury commentary for the sponsor: (a) position \
and covenant summary, (b) the 13-week path and the trough, (c) what the probabilistic \
view adds and the breach risk, (d) recommended actions with their effect on covenant \
headroom. Give each section a confidence level and open flags.

9. submit_for_review -- Send the narrative and your single recommended treasury \
action to the human reviewer. DO NOT proceed until the reviewer responds. This is a \
mandatory gate. The agent proposes; the human disposes.

10. log_output -- After the reviewer's disposition, write the complete audit trail.

RULES FOR COMMENTARY

- Write for a private equity sponsor and a lender. Use treasury terminology: \
availability, headroom, borrowing base, draw, sweep, DSO, debt service.
- Be specific. "Cash gets tight in the middle of the forecast" is not useful. "Cash \
troughs at $1.6 million in week 8 after a $250,000 revolver draw in week 7, driven by \
the Vantage Aerospace collection slipping from week 4 to week 7" is useful.
- Cite the driver behind every number. Tie the trough to the collections and \
disbursements that create it.
- Distinguish timing from structural. A customer paying in 65 days instead of 45 is \
timing. A money-losing program is structural. The sponsor needs to know which is which.
- Present the deterministic and probabilistic views together. If the single-point \
forecast passes the covenant but the distribution shows material breach risk, say so \
plainly and quantify it.
- Do not speculate beyond what the data supports. If you cannot attribute a risk, \
say so explicitly.
- Do not use em dashes. Vary sentence lengths. Write in a direct, professional \
register."""

INITIAL_PROMPT = """Build Cascade Precision Products' 13-week cash flow forecast for \
Granite Peak Partners. Load the facility and position, load the receivables, load the \
payables and fixed items, build the deterministic roll-forward, run the Monte Carlo, \
run the sensitivity sweep and the scenarios you judge most relevant to the covenant, \
compute the trailing 4-week variance, draft the treasury narrative with a single \
recommended action, and submit it for human review. Proceed step by step using your \
tools."""


## 1.2 · `engine.py` — the deterministic core

A pure function: position + dated receipts + dated disbursements in, 13 weekly rows out.
Per week: beginning cash + receipts − disbursements = pre-revolver cash, then the
revolver mechanics every treasury team runs by hand:

- **Below the \$1.5M floor** → auto-draw the revolver (rounded up to \$250K increments,
  capped at availability) to restore it.
- **Comfortably above the floor** (floor + \$500K buffer) → sweep the excess to pay the
  revolver down.

It reports, per week: receipts, disbursements, pre-revolver cash, draw/paydown, revolver
balance, ending cash, and **covenant liquidity** (cash + undrawn availability) — plus the
trough week and week-13 covenant headroom.

Two design points worth pausing on:

1. **Collections are forecast from behavior, not stated terms.** `expected_pay_date()`
   uses each customer's observed days-to-pay. Vantage Aerospace is on Net 45 and pays in
   ~64 days; that 19-day gap is what moves its \$2.1M milestone from the week-4 pencil to
   a week-7 landing.
2. **No side effects, no randomness, no I/O.** The Monte Carlo layer calls this function
   a thousand times; purity is what makes that a one-liner.

In [ ]:
%%writefile cashflow_harness/engine.py
"""Deterministic 13-week cash roll-forward with revolver mechanics.

Everything in this module is a pure function of its inputs. No file I/O, no
global state, no randomness. The Monte Carlo layer calls run_forecast()
thousands of times with resampled flows; it must stay side-effect free.

Week convention: weeks are indexed 1..num_weeks. Week 1 begins the Monday
after the latest Friday close. Dates are ISO strings ("YYYY-MM-DD");
date_to_week() maps a date onto the week grid.
"""

import math
from datetime import date, timedelta


def week_starts(week1_monday: date, num_weeks: int = 13) -> list[date]:
    """The Monday that opens each week, weeks 1..num_weeks."""
    return [week1_monday + timedelta(weeks=i) for i in range(num_weeks)]


def date_to_week(d: date, week1_monday: date, num_weeks: int = 13) -> int:
    """Map a date to a week index 1..num_weeks.

    Dates before the horizon clamp to week 1 (an overdue receipt or payment
    lands in the first week). Dates past the horizon return num_weeks + 1 so
    the caller can drop them from the 13-week view.
    """
    offset = (d - week1_monday).days
    if offset < 0:
        return 1
    week = offset // 7 + 1
    return week if week <= num_weeks else num_weeks + 1


def bucket_flows(
    items: list[dict],
    week1_monday: date,
    num_weeks: int = 13,
    date_key: str = "date",
    amount_key: str = "amount",
) -> list[float]:
    """Sum item amounts into weekly buckets. Returns a list of num_weeks floats.

    Items dated past the horizon fall out of the window (they are tail
    collections or payments beyond week 13).
    """
    weekly = [0.0] * num_weeks
    for item in items:
        d = item[date_key]
        if isinstance(d, str):
            d = date.fromisoformat(d)
        w = date_to_week(d, week1_monday, num_weeks)
        if w <= num_weeks:
            weekly[w - 1] += float(item[amount_key])
    return weekly


def run_forecast(
    beginning_cash: float,
    revolver_limit: float,
    revolver_drawn: float,
    operating_floor: float,
    covenant_threshold: float,
    weekly_receipts: list[float],
    weekly_disbursements: list[float],
    sweep_buffer: float = 500_000,
    draw_increment: float = 250_000,
    covenant_test_week: int = 13,
) -> dict:
    """Roll cash forward week by week and apply revolver mechanics.

    Per week: beginning cash + receipts - disbursements = pre-revolver cash.
    If pre-revolver cash falls below the operating floor, auto-draw on the
    revolver (rounded up to draw_increment, capped at availability) to restore
    the floor. If cash sits above floor + sweep_buffer and the revolver is
    drawn, sweep the excess to pay it down.

    Covenant liquidity is ending cash plus undrawn availability, tested at
    covenant_test_week.
    """
    num_weeks = len(weekly_receipts)
    assert len(weekly_disbursements) == num_weeks

    cash = float(beginning_cash)
    revolver = float(revolver_drawn)
    weeks = []

    for i in range(num_weeks):
        receipts = float(weekly_receipts[i])
        disbursements = float(weekly_disbursements[i])
        pre_revolver = cash + receipts - disbursements

        draw = 0.0
        paydown = 0.0
        availability = revolver_limit - revolver

        if pre_revolver < operating_floor:
            shortfall = operating_floor - pre_revolver
            draw = math.ceil(shortfall / draw_increment) * draw_increment
            draw = min(draw, availability)
        elif pre_revolver > operating_floor + sweep_buffer and revolver > 0:
            paydown = min(revolver, pre_revolver - (operating_floor + sweep_buffer))

        revolver = revolver + draw - paydown
        ending_cash = pre_revolver + draw - paydown
        availability = revolver_limit - revolver
        liquidity = ending_cash + availability

        weeks.append({
            "week": i + 1,
            "receipts": round(receipts, 2),
            "disbursements": round(disbursements, 2),
            "net_flow": round(receipts - disbursements, 2),
            "pre_revolver_cash": round(pre_revolver, 2),
            "revolver_draw": round(draw, 2),
            "revolver_paydown": round(paydown, 2),
            "revolver_balance": round(revolver, 2),
            "availability": round(availability, 2),
            "ending_cash": round(ending_cash, 2),
            "covenant_liquidity": round(liquidity, 2),
        })

        cash = ending_cash

    trough = min(weeks, key=lambda w: w["ending_cash"])
    test_week = weeks[covenant_test_week - 1]
    headroom = test_week["covenant_liquidity"] - covenant_threshold

    return {
        "weeks": weeks,
        "trough_week": trough["week"],
        "trough_cash": trough["ending_cash"],
        "min_covenant_liquidity": min(w["covenant_liquidity"] for w in weeks),
        "covenant_test_week": covenant_test_week,
        "covenant_threshold": covenant_threshold,
        "test_week_liquidity": test_week["covenant_liquidity"],
        "covenant_headroom": round(headroom, 2),
        "covenant_pass": headroom >= 0,
        "below_floor_any_week": any(
            w["pre_revolver_cash"] < operating_floor for w in weeks
        ),
        "total_revolver_draws": round(sum(w["revolver_draw"] for w in weeks), 2),
        "ending_revolver_balance": weeks[-1]["revolver_balance"],
    }


def expected_pay_date(invoice: dict) -> date:
    """Deterministic expected collection date for an open invoice.

    Uses the customer's observed payment behavior (historical average days to
    pay from invoice date), not the stated terms. That gap is the point: the
    forecast reflects how customers actually pay.
    """
    inv_date = invoice["invoice_date"]
    if isinstance(inv_date, str):
        inv_date = date.fromisoformat(inv_date)
    return inv_date + timedelta(days=int(round(invoice["customer_days_to_pay"])))


# A billing week does not collect as one lump. Customers pay across a
# spread; model each week's collections as three tranches around the lag.
COLLECTION_TRANCHES = [(0.25, -7.0), (0.50, 0.0), (0.25, 7.0)]


def split_sales_row(row: dict) -> list[tuple[float, float]]:
    """Split a forecast billing week into (amount, collection_lag_days) parts.

    Vantage Aerospace's share of new billings collects at Vantage's observed
    pay behavior, not the blended lag. Its stretch therefore pushes part of
    the late-quarter billings past the covenant test week, which is exactly
    the exposure the accelerate-Vantage scenario removes.
    """
    billings = float(row["billings"])
    vantage_share = float(row.get("vantage_share", 0.0))
    vantage_amt = billings * vantage_share
    parts = []
    for weight, offset in COLLECTION_TRANCHES:
        parts.append(((billings - vantage_amt) * weight, float(row["base_lag_days"]) + offset, "base"))
        if vantage_amt > 0:
            parts.append((vantage_amt * weight, float(row["vantage_lag_days"]) + offset, "vantage"))
    return parts


def build_weekly_flows(
    invoices: list[dict],
    bills: list[dict],
    fixed_items: list[dict],
    sales_forecast: list[dict],
    week1_monday: date,
    num_weeks: int = 13,
) -> tuple[list[float], list[float]]:
    """Assemble the deterministic weekly receipt and disbursement vectors.

    Receipts: open invoices at their expected pay dates, plus collections on
    forecast new billings (billing week start + the blended collection lag).
    Disbursements: open AP at due dates, plus every dated fixed-schedule item.
    """
    ar_items = [
        {"date": expected_pay_date(inv), "amount": inv["amount"]} for inv in invoices
    ]
    receipts = bucket_flows(ar_items, week1_monday, num_weeks)

    for row in sales_forecast:
        for amount, lag_days, _source in split_sales_row(row):
            billing_date = row["week_start"]
            if isinstance(billing_date, str):
                billing_date = date.fromisoformat(billing_date)
            collect_date = billing_date + timedelta(days=int(lag_days))
            w = date_to_week(collect_date, week1_monday, num_weeks)
            if w <= num_weeks:
                receipts[w - 1] += amount

    ap_items = [{"date": b["due_date"], "amount": b["amount"]} for b in bills]
    disbursements = bucket_flows(ap_items, week1_monday, num_weeks)

    fixed = bucket_flows(fixed_items, week1_monday, num_weeks)
    disbursements = [d + f for d, f in zip(disbursements, fixed)]

    return receipts, disbursements


## 1.3 · `montecarlo.py` — the probabilistic layer

This is the module that makes the build unusual, so name the logic plainly: the
deterministic forecast in 1.2 collects every invoice at its customer's *average*
observed behavior. But nobody pays at their average every time — the average came with
a variance, measured from the same history. If the mean of observed behavior belongs in
the forecast, its variance does too. Refusing the variance doesn't make the risk go
away; it just makes it invisible.

So each iteration re-answers one question: *what if customers pay the way they actually
pay, and the ramp bills what it actually bills?* Then it re-runs the deterministic
engine — a thousand times.

The sampling model is deliberately structured, not just noise:

- **Collection timing** — each customer gets one *systematic* shift per iteration (a
  customer that pays late pays late on every invoice; 60% of its sigma) plus
  *idiosyncratic* per-invoice noise (80% of sigma). The split preserves total observed
  variance while creating the correlated risk that actually threatens covenants.
- **Sales volume** — each week's new billings draw around the forecast with that week's
  sigma (rising 10% → 15% into the ramp, because new programs are less certain).
- **Collection tranches** — a billing week doesn't collect as one lump; it spreads
  25/50/25 around the blended lag. That keeps the exposure at the week-13 boundary smooth
  and the simulation median centered on the deterministic path.

Outputs: P10/P50/P90 bands per week, the trough distribution, week-13 liquidity
percentiles, **P(covenant breach at week 13)** — the number that changes the meeting —
and P(the floor cannot be restored within the revolver limit), which is the treasury
definition of real trouble.

In [ ]:
%%writefile cashflow_harness/montecarlo.py
"""Probabilistic layer: Monte Carlo over collection timing and sales volume.

Each iteration resamples when customers actually pay and how much the ramp
actually bills, rebuilds the weekly flows, and re-runs the deterministic
engine. The engine is pure, so a thousand replays are just a thousand
function calls.

Sampling model:
  - Collection timing. Each customer's days-to-pay gets one systematic shift
    per iteration (a customer that pays late pays late on every invoice),
    plus idiosyncratic noise per invoice. The two components split the
    customer's observed sigma so total variance matches history.
  - Sales volume. Each week's new billings draw around the forecast with
    that week's sigma. Collections keep the blended lag; the lag itself also
    gets a small systematic shift per iteration, because the ramp billings
    collecting near the week-13 boundary are exactly where timing risk turns
    into covenant risk.

Reported "below floor" probability means ending cash below the operating
floor even after revolver draws, i.e. the draw needed to restore the floor
exceeded remaining availability. That is the treasury definition of trouble:
the floor cannot be bought back.
"""

from datetime import date, timedelta

import numpy as np

from . import engine

SYSTEMATIC_SHARE = 0.6      # share of pay-time sigma that is per-customer, per-iteration
BASE_LAG_SIGMA_DAYS = 3     # systematic shift on the blended new-billings lag
WEEK_LAG_SIGMA_DAYS = 4     # additional per-billing-week lag noise


def run_monte_carlo(
    invoices: list[dict],
    bills: list[dict],
    fixed_items: list[dict],
    sales_forecast: list[dict],
    facility: dict,
    iterations: int = 1_000,
    seed: int = 42,
) -> dict:
    rng = np.random.default_rng(seed)
    week1_monday = date.fromisoformat(facility["week1_monday"])
    num_weeks = int(facility.get("num_weeks", 13))
    floor = facility["operating_cash_floor"]
    threshold = facility["covenant"]["threshold"]

    # Disbursements do not vary across iterations.
    ap_items = [{"date": b["due_date"], "amount": b["amount"]} for b in bills]
    disbursements = engine.bucket_flows(ap_items, week1_monday, num_weeks)
    fixed = engine.bucket_flows(fixed_items, week1_monday, num_weeks)
    disbursements = [d + f for d, f in zip(disbursements, fixed)]

    customers = sorted({inv["customer"] for inv in invoices})
    cust_sigma = {
        c: max(inv["customer_pay_sigma"] for inv in invoices if inv["customer"] == c)
        for c in customers
    }

    idio_share = (1 - SYSTEMATIC_SHARE**2) ** 0.5

    ending_cash = np.zeros((iterations, num_weeks))
    liquidity = np.zeros((iterations, num_weeks))
    troughs = np.zeros(iterations)
    trough_weeks = np.zeros(iterations, dtype=int)
    below_floor = np.zeros(iterations, dtype=bool)
    breach = np.zeros(iterations, dtype=bool)

    for it in range(iterations):
        shifts = {
            c: rng.normal(0.0, SYSTEMATIC_SHARE * cust_sigma[c]) for c in customers
        }
        base_lag_shift = rng.normal(0.0, BASE_LAG_SIGMA_DAYS)

        receipts = [0.0] * num_weeks
        for inv in invoices:
            days = (
                inv["customer_days_to_pay"]
                + shifts[inv["customer"]]
                + rng.normal(0.0, idio_share * inv["customer_pay_sigma"])
            )
            pay = date.fromisoformat(inv["invoice_date"]) + timedelta(days=int(round(max(days, 1))))
            w = engine.date_to_week(pay, week1_monday, num_weeks)
            if w <= num_weeks:
                receipts[w - 1] += inv["amount"]

        vantage_shift = shifts.get("Vantage Aerospace", 0.0)
        for row in sales_forecast:
            billings = max(rng.normal(row["billings"], row["sigma"]), 0.0)
            week_start = date.fromisoformat(str(row["week_start"]))
            week_lag_noise = rng.normal(0.0, WEEK_LAG_SIGMA_DAYS)
            for amount, lag, source in engine.split_sales_row(dict(row, billings=billings)):
                if amount <= 0:
                    continue
                shift = vantage_shift if source == "vantage" else base_lag_shift + week_lag_noise
                collect = week_start + timedelta(days=int(round(lag + shift)))
                w = engine.date_to_week(collect, week1_monday, num_weeks)
                if w <= num_weeks:
                    receipts[w - 1] += amount

        result = engine.run_forecast(
            beginning_cash=facility["beginning_cash"],
            revolver_limit=facility["revolver_limit"],
            revolver_drawn=facility["revolver_drawn"],
            operating_floor=floor,
            covenant_threshold=threshold,
            weekly_receipts=receipts,
            weekly_disbursements=disbursements,
            sweep_buffer=facility.get("sweep_buffer", 500_000),
            draw_increment=facility.get("draw_increment", 250_000),
            covenant_test_week=facility["covenant"]["test_week"],
        )

        weeks = result["weeks"]
        ending_cash[it] = [w["ending_cash"] for w in weeks]
        liquidity[it] = [w["covenant_liquidity"] for w in weeks]
        troughs[it] = result["trough_cash"]
        trough_weeks[it] = result["trough_week"]
        below_floor[it] = any(w["ending_cash"] < floor - 0.01 for w in weeks)
        breach[it] = not result["covenant_pass"]

    def pct(arr, q):
        return np.percentile(arr, q, axis=0)

    test_ix = facility["covenant"]["test_week"] - 1
    w13_liq = liquidity[:, test_ix]

    return {
        "iterations": iterations,
        "seed": seed,
        "weeks": list(range(1, num_weeks + 1)),
        "ending_cash_p10": [round(x) for x in pct(ending_cash, 10)],
        "ending_cash_p50": [round(x) for x in pct(ending_cash, 50)],
        "ending_cash_p90": [round(x) for x in pct(ending_cash, 90)],
        "liquidity_p10": [round(x) for x in pct(liquidity, 10)],
        "liquidity_p50": [round(x) for x in pct(liquidity, 50)],
        "liquidity_p90": [round(x) for x in pct(liquidity, 90)],
        "trough_cash_p10": round(float(np.percentile(troughs, 10))),
        "trough_cash_p50": round(float(np.percentile(troughs, 50))),
        "trough_cash_p90": round(float(np.percentile(troughs, 90))),
        "trough_week_mode": int(np.bincount(trough_weeks).argmax()),
        "test_week": facility["covenant"]["test_week"],
        "test_week_liquidity_p10": round(float(np.percentile(w13_liq, 10))),
        "test_week_liquidity_p50": round(float(np.percentile(w13_liq, 50))),
        "test_week_liquidity_p90": round(float(np.percentile(w13_liq, 90))),
        "covenant_threshold": threshold,
        "operating_floor": floor,
        "p_below_floor_any_week": round(float(below_floor.mean()), 4),
        "p_covenant_breach": round(float(breach.mean()), 4),
    }


## 1.4 · The fixed dataset — Cascade Precision's book of record

Synthetic, but not arbitrary — and **frozen**. These are the exact files the demo was
calibrated and tested against, embedded byte for byte: they were produced once by the
repo's seeded generator (`cashflow_harness/data_gen.py`, anchored to the 2026-08-21
Friday close) and never regenerated at run time, so there is no drift risk between what
was tested and what runs. The control totals: open AR of exactly \$19.3M across 122
invoices with **Vantage Aerospace at \$4.2M (21.8%)**, open AP of exactly \$8.7M across
90 bills, every scheduled one-timer in its week, and trailing 4-week actuals exactly
6.0% under the prior forecast.

Five stories are baked in for the agent to surface:

1. **The Vantage term stretch** — the largest customer slid from Net 45 to ~64 days;
   a \$2.1M collection penciled for week 4 now lands week 7.
2. **The growth working-capital build** — ramp material buys and a second shift hit
   weeks 3–7, ahead of billings that collect weeks 9–13 and beyond.
3. **Lumpy scheduled outflows** — insurance (wk 5), capex deposit (wk 6), earn-out
   (wk 9), tax (wk 10), \$2.2M debt service (wk 13).
4. **A tight-but-passing covenant** — trough ~\$1.56M in week 8, week-13 headroom ~\$631K.
5. **The probabilistic reversal** — P50 passes, P10 breaches; ~20% breach probability.

Seven files: the treasury context memo, the facility terms, the AR and AP ledgers (one
record per line), the fixed disbursement schedule, the sales forecast, and the trailing
actuals. Part 2.1 verifies every control total before anything else runs; the design
reference lives in the repo as `data/DATA_DESIGN.md`.

In [ ]:
%%writefile data/company_context.md
# Cascade Precision Products, Inc. -- Treasury Context

As of the Friday close on 2026-08-21. Forecast horizon: 13 weeks
beginning Monday 2026-08-24.

## Ownership and the ask

Granite Peak Partners closed its leveraged buyout of Cascade Precision ten
weeks ago. The capital structure is a $38.0M term loan and a $15.0M revolver
($6.0M drawn at the close date), both agented by Ridgeline Capital Bank. The
sponsor's first standing request is a weekly 13-week cash flow forecast with
covenant visibility. This is the first one under new ownership.

## The covenant

Minimum liquidity (cash plus undrawn revolver availability) of $4.0M, measured
weekly, hard-tested and certified to the lender at the week 13 quarter-end. A
breach is an event of default under the credit agreement. Quarterly term-loan
debt service of $2.2M ($1.25M amortization, $0.95M interest) is due at the end
of week 13.

Treasury policy: operating cash floor of $1.5M. If ending cash would fall
below the floor, draw the revolver (in $250K increments) to restore it. When
cash runs comfortably above the floor, sweep the excess against the revolver,
leaving a $500K buffer.

## Customer concentration and the Vantage term stretch

Vantage Aerospace is the largest account at roughly 22% of open AR ($4.2M).
Vantage is on Net 45 terms but its payment behavior has stretched to roughly
64 days over the last two quarters as its own program payments slowed. The
practical effect: a $2.1M milestone collection on the LR-7 landing gear
program that the prior forecast penciled for week 4 (its contractual due
date) is now expected to land in week 7 or 8. Vantage remains a sound credit;
this is timing, not collectability. Procurement at Vantage has been
non-committal about acceleration but has paid to a negotiated date before
when offered a modest early-pay incentive.

The top five customers (Vantage Aerospace, Trident Defense Systems, Pacific
Turbine Group, Halvorsen Industrial, Bellingham Aerostructures) are about 60%
of open AR.

## The aerospace ramp

Cascade won a multi-year aerospace program earlier this year. Shipments ramp
through the quarter: material buys and outside processing are already flowing
(heavier AP due weeks 3 through 7), a second shift was added (payroll steps
up from the week 5 cycle), and the associated billings build from week 6
onward, collecting on normal lags in weeks 9 through 13 and beyond. Cash goes
out before it comes in. This is deliberate growth working capital, not
deterioration.

## Scheduled one-time outflows

- Week 5: semiannual property & casualty insurance premium, $450K
- Week 6: capex deposit on a Hermle C42 5-axis machining center, $800K
  (progress deposit; the builder has flexibility on the deposit date within
  the quarter, and slots the machine for delivery next quarter)
- Week 9: seller note earn-out payment from the acquisition, $1.0M (fixed
  date per the purchase agreement)
- Week 10: estimated federal income tax payment, $300K
- Week 13: term loan quarterly debt service, $2.2M

## Trailing performance

Over the trailing 4 weeks, collections came in about 6% under the prior
forecast. The miss is concentrated in one week and one customer: a Vantage
invoice of roughly $0.4M slipped past its forecast date. Disbursements ran
close to plan. This is the same behavior now baked into the week 4 to week 7
slip on the $2.1M milestone.

## Notes for the forecast

- The borrowing base comfortably exceeds the revolver commitment through the
  horizon; availability is constrained by the $15.0M limit, not the base.
- Payroll is biweekly (weeks 1, 3, 5, 7, 9, 11, 13), stepping up when the
  second shift loads in.
- The capex deposit and a small set of AP items are flagged discretionary;
  everything else is committed.


In [ ]:
%%writefile data/facility_terms.json
{
  "company": "Cascade Precision Products, Inc.",
  "as_of_close": "2026-08-21",
  "week1_monday": "2026-08-24",
  "num_weeks": 13,
  "beginning_cash": 2400000,
  "revolver_limit": 15000000,
  "revolver_drawn": 6000000,
  "revolver_availability": 9000000,
  "operating_cash_floor": 1500000,
  "sweep_buffer": 500000,
  "draw_increment": 250000,
  "covenant": {
    "type": "minimum liquidity (cash + undrawn revolver availability)",
    "threshold": 4000000,
    "test_week": 13,
    "consequence": "event of default",
    "certified_to": "Ridgeline Capital Bank, agent for the lender group"
  },
  "term_loan": {
    "balance": 38000000,
    "quarterly_amortization": 1250000,
    "quarterly_interest": 950000,
    "debt_service_due_week": 13
  }
}

In [ ]:
%%writefile data/ar_open_invoices.json
[
{"invoice_id": "INV-5001", "customer": "Vantage Aerospace", "segment": "aerospace", "invoice_date": "2026-08-02", "due_date": "2026-09-16", "terms": "Net 45", "amount": 2100000, "customer_days_to_pay": 64, "customer_pay_sigma": 10.0, "memo": "Milestone billing, LR-7 landing gear program"},
{"invoice_id": "INV-5002", "customer": "Vantage Aerospace", "segment": "aerospace", "invoice_date": "2026-06-27", "due_date": "2026-08-11", "terms": "Net 45", "amount": 620000, "customer_days_to_pay": 64, "customer_pay_sigma": 10.0, "memo": "Production release 4180, actuator housings"},
{"invoice_id": "INV-5003", "customer": "Vantage Aerospace", "segment": "aerospace", "invoice_date": "2026-07-12", "due_date": "2026-08-26", "terms": "Net 45", "amount": 560000, "customer_days_to_pay": 64, "customer_pay_sigma": 10.0, "memo": "Production release 4197, actuator housings"},
{"invoice_id": "INV-5004", "customer": "Vantage Aerospace", "segment": "aerospace", "invoice_date": "2026-07-22", "due_date": "2026-09-05", "terms": "Net 45", "amount": 480000, "customer_days_to_pay": 64, "customer_pay_sigma": 10.0, "memo": "Tooling amortization billing, LR-7"},
{"invoice_id": "INV-5005", "customer": "Vantage Aerospace", "segment": "aerospace", "invoice_date": "2026-08-11", "due_date": "2026-09-25", "terms": "Net 45", "amount": 440000, "customer_days_to_pay": 64, "customer_pay_sigma": 10.0, "memo": "Production release 4211, wing rib fittings"},
{"invoice_id": "INV-5006", "customer": "Ridgefield Turbine Services", "segment": "industrial", "invoice_date": "2026-07-31", "due_date": "2026-09-29", "terms": "Net 60", "amount": 127900, "customer_days_to_pay": 115, "customer_pay_sigma": 12.0, "memo": ""},
{"invoice_id": "INV-5007", "customer": "Ridgefield Turbine Services", "segment": "industrial", "invoice_date": "2026-08-01", "due_date": "2026-09-30", "terms": "Net 60", "amount": 169000, "customer_days_to_pay": 115, "customer_pay_sigma": 12.0, "memo": ""},
{"invoice_id": "INV-5008", "customer": "Ridgefield Turbine Services", "segment": "industrial", "invoice_date": "2026-08-01", "due_date": "2026-09-30", "terms": "Net 60", "amount": 150600, "customer_days_to_pay": 115, "customer_pay_sigma": 12.0, "memo": ""},
{"invoice_id": "INV-5009", "customer": "Ridgefield Turbine Services", "segment": "industrial", "invoice_date": "2026-08-04", "due_date": "2026-10-03", "terms": "Net 60", "amount": 67500, "customer_days_to_pay": 115, "customer_pay_sigma": 12.0, "memo": ""},
{"invoice_id": "INV-5010", "customer": "Sturgeon Bay Marine Systems", "segment": "industrial", "invoice_date": "2026-08-03", "due_date": "2026-10-02", "terms": "Net 60", "amount": 31800, "customer_days_to_pay": 112, "customer_pay_sigma": 10.0, "memo": ""},
{"invoice_id": "INV-5011", "customer": "Sturgeon Bay Marine Systems", "segment": "industrial", "invoice_date": "2026-08-05", "due_date": "2026-10-04", "terms": "Net 60", "amount": 145800, "customer_days_to_pay": 112, "customer_pay_sigma": 10.0, "memo": ""},
{"invoice_id": "INV-5012", "customer": "Sturgeon Bay Marine Systems", "segment": "industrial", "invoice_date": "2026-08-07", "due_date": "2026-10-06", "terms": "Net 60", "amount": 142400, "customer_days_to_pay": 112, "customer_pay_sigma": 10.0, "memo": ""},
{"invoice_id": "INV-5013", "customer": "Saginaw Rail Components", "segment": "industrial", "invoice_date": "2026-08-17", "due_date": "2026-10-16", "terms": "Net 60", "amount": 123300, "customer_days_to_pay": 78, "customer_pay_sigma": 11.0, "memo": ""},
{"invoice_id": "INV-5014", "customer": "Saginaw Rail Components", "segment": "industrial", "invoice_date": "2026-08-18", "due_date": "2026-10-17", "terms": "Net 60", "amount": 117500, "customer_days_to_pay": 78, "customer_pay_sigma": 11.0, "memo": ""},
{"invoice_id": "INV-5015", "customer": "Saginaw Rail Components", "segment": "industrial", "invoice_date": "2026-08-18", "due_date": "2026-10-17", "terms": "Net 60", "amount": 164200, "customer_days_to_pay": 78, "customer_pay_sigma": 11.0, "memo": ""},
{"invoice_id": "INV-5016", "customer": "Blue Ridge Turbomachinery", "segment": "aerospace", "invoice_date": "2026-08-20", "due_date": "2026-10-19", "terms": "Net 60", "amount": 105700, "customer_days_to_pay": 67, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5017", "customer": "Blue Ridge Turbomachinery", "segment": "aerospace", "invoice_date": "2026-08-20", "due_date": "2026-10-19", "terms": "Net 60", "amount": 165900, "customer_days_to_pay": 67, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5018", "customer": "Blue Ridge Turbomachinery", "segment": "aerospace", "invoice_date": "2026-08-20", "due_date": "2026-10-19", "terms": "Net 60", "amount": 138300, "customer_days_to_pay": 67, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5019", "customer": "Blue Ridge Turbomachinery", "segment": "aerospace", "invoice_date": "2026-08-15", "due_date": "2026-10-14", "terms": "Net 60", "amount": 115100, "customer_days_to_pay": 67, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5020", "customer": "Blue Ridge Turbomachinery", "segment": "aerospace", "invoice_date": "2026-08-14", "due_date": "2026-10-13", "terms": "Net 60", "amount": 165000, "customer_days_to_pay": 67, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5021", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-18", "due_date": "2026-10-17", "terms": "Net 60", "amount": 88500, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5022", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-15", "due_date": "2026-10-14", "terms": "Net 60", "amount": 193100, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5023", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-21", "due_date": "2026-10-20", "terms": "Net 60", "amount": 61600, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5024", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-17", "due_date": "2026-10-16", "terms": "Net 60", "amount": 59700, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5025", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-15", "due_date": "2026-10-14", "terms": "Net 60", "amount": 170500, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5026", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-16", "due_date": "2026-10-15", "terms": "Net 60", "amount": 159300, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5027", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-14", "due_date": "2026-10-13", "terms": "Net 60", "amount": 263600, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5028", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-10", "due_date": "2026-10-09", "terms": "Net 60", "amount": 197000, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5029", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-11", "due_date": "2026-10-10", "terms": "Net 60", "amount": 170300, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5030", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-10", "due_date": "2026-10-09", "terms": "Net 60", "amount": 166300, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5031", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-07", "due_date": "2026-10-06", "terms": "Net 60", "amount": 108700, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5032", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-09", "due_date": "2026-10-08", "terms": "Net 60", "amount": 54100, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5033", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-08", "due_date": "2026-10-07", "terms": "Net 60", "amount": 95900, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5034", "customer": "Pacific Turbine Group", "segment": "aerospace", "invoice_date": "2026-08-11", "due_date": "2026-10-10", "terms": "Net 60", "amount": 211400, "customer_days_to_pay": 66, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5035", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-12", "due_date": "2026-10-11", "terms": "Net 60", "amount": 123900, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5036", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-11", "due_date": "2026-10-10", "terms": "Net 60", "amount": 181000, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5037", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-12", "due_date": "2026-10-11", "terms": "Net 60", "amount": 145900, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5038", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-04", "due_date": "2026-10-03", "terms": "Net 60", "amount": 163200, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5039", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-02", "due_date": "2026-10-01", "terms": "Net 60", "amount": 53100, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5040", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-04", "due_date": "2026-10-03", "terms": "Net 60", "amount": 129200, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5041", "customer": "Cobalt Marine Propulsion", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-10-02", "terms": "Net 60", "amount": 123700, "customer_days_to_pay": 65, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5042", "customer": "Tectonic Mining Systems", "segment": "industrial", "invoice_date": "2026-08-05", "due_date": "2026-09-19", "terms": "Net 45", "amount": 64700, "customer_days_to_pay": 55, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5043", "customer": "Tectonic Mining Systems", "segment": "industrial", "invoice_date": "2026-08-04", "due_date": "2026-09-18", "terms": "Net 45", "amount": 140100, "customer_days_to_pay": 55, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5044", "customer": "Tectonic Mining Systems", "segment": "industrial", "invoice_date": "2026-08-05", "due_date": "2026-09-19", "terms": "Net 45", "amount": 125200, "customer_days_to_pay": 55, "customer_pay_sigma": 8.0, "memo": ""},
{"invoice_id": "INV-5045", "customer": "Whitewater Compressor", "segment": "industrial", "invoice_date": "2026-08-16", "due_date": "2026-09-30", "terms": "Net 45", "amount": 41400, "customer_days_to_pay": 54, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5046", "customer": "Whitewater Compressor", "segment": "industrial", "invoice_date": "2026-08-07", "due_date": "2026-09-21", "terms": "Net 45", "amount": 82700, "customer_days_to_pay": 54, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5047", "customer": "Whitewater Compressor", "segment": "industrial", "invoice_date": "2026-08-08", "due_date": "2026-09-22", "terms": "Net 45", "amount": 55900, "customer_days_to_pay": 54, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5048", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-11", "due_date": "2026-09-25", "terms": "Net 45", "amount": 190600, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5049", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-07", "due_date": "2026-09-21", "terms": "Net 45", "amount": 199100, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5050", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-09", "due_date": "2026-09-23", "terms": "Net 45", "amount": 82700, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5051", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-11", "due_date": "2026-09-25", "terms": "Net 45", "amount": 146800, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5052", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-11", "due_date": "2026-09-25", "terms": "Net 45", "amount": 102300, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5053", "customer": "Ironwood Energy Equipment", "segment": "industrial", "invoice_date": "2026-08-08", "due_date": "2026-09-22", "terms": "Net 45", "amount": 138500, "customer_days_to_pay": 52, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5054", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-08", "due_date": "2026-09-22", "terms": "Net 45", "amount": 155800, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5055", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-09", "due_date": "2026-09-23", "terms": "Net 45", "amount": 91100, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5056", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-12", "due_date": "2026-09-26", "terms": "Net 45", "amount": 191000, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5057", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-10", "due_date": "2026-09-24", "terms": "Net 45", "amount": 154100, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5058", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-09", "due_date": "2026-09-23", "terms": "Net 45", "amount": 61600, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5059", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-02", "due_date": "2026-09-16", "terms": "Net 45", "amount": 185900, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5060", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-04", "due_date": "2026-09-18", "terms": "Net 45", "amount": 203300, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5061", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-01", "due_date": "2026-09-15", "terms": "Net 45", "amount": 196200, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5062", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-05", "due_date": "2026-09-19", "terms": "Net 45", "amount": 138000, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5063", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-12", "due_date": "2026-09-26", "terms": "Net 45", "amount": 64100, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5064", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 72200, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5065", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 200400, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5066", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-04", "due_date": "2026-09-18", "terms": "Net 45", "amount": 134900, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5067", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 70200, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5068", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 192700, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5069", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-02", "due_date": "2026-09-16", "terms": "Net 45", "amount": 150500, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5070", "customer": "Trident Defense Systems", "segment": "defense", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 138000, "customer_days_to_pay": 51, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5071", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-08-05", "due_date": "2026-09-19", "terms": "Net 45", "amount": 50100, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5072", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-08-02", "due_date": "2026-09-16", "terms": "Net 45", "amount": 120400, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5073", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-08-04", "due_date": "2026-09-18", "terms": "Net 45", "amount": 51200, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5074", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-08-03", "due_date": "2026-09-17", "terms": "Net 45", "amount": 69900, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5075", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-29", "due_date": "2026-09-12", "terms": "Net 45", "amount": 240900, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5076", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-30", "due_date": "2026-09-13", "terms": "Net 45", "amount": 178200, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5077", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-27", "due_date": "2026-09-10", "terms": "Net 45", "amount": 131700, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5078", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-30", "due_date": "2026-09-13", "terms": "Net 45", "amount": 151100, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5079", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-28", "due_date": "2026-09-11", "terms": "Net 45", "amount": 221800, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5080", "customer": "Bellingham Aerostructures", "segment": "aerospace", "invoice_date": "2026-07-28", "due_date": "2026-09-11", "terms": "Net 45", "amount": 114700, "customer_days_to_pay": 50, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5081", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-30", "due_date": "2026-09-13", "terms": "Net 45", "amount": 187200, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5082", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-23", "due_date": "2026-09-06", "terms": "Net 45", "amount": 61800, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5083", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-20", "due_date": "2026-09-03", "terms": "Net 45", "amount": 191200, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5084", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-24", "due_date": "2026-09-07", "terms": "Net 45", "amount": 37600, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5085", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-22", "due_date": "2026-09-05", "terms": "Net 45", "amount": 161300, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5086", "customer": "Northgate Avionics", "segment": "aerospace", "invoice_date": "2026-07-24", "due_date": "2026-09-07", "terms": "Net 45", "amount": 170900, "customer_days_to_pay": 49, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5087", "customer": "Deschutes Fabrication", "segment": "industrial", "invoice_date": "2026-07-31", "due_date": "2026-08-30", "terms": "Net 30", "amount": 11200, "customer_days_to_pay": 42, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5088", "customer": "Deschutes Fabrication", "segment": "industrial", "invoice_date": "2026-07-31", "due_date": "2026-08-30", "terms": "Net 30", "amount": 40500, "customer_days_to_pay": 42, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5089", "customer": "Deschutes Fabrication", "segment": "industrial", "invoice_date": "2026-07-29", "due_date": "2026-08-28", "terms": "Net 30", "amount": 48300, "customer_days_to_pay": 42, "customer_pay_sigma": 7.0, "memo": ""},
{"invoice_id": "INV-5090", "customer": "Orchard Robotics", "segment": "industrial", "invoice_date": "2026-08-01", "due_date": "2026-08-31", "terms": "Net 30", "amount": 130400, "customer_days_to_pay": 41, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5091", "customer": "Orchard Robotics", "segment": "industrial", "invoice_date": "2026-07-29", "due_date": "2026-08-28", "terms": "Net 30", "amount": 61700, "customer_days_to_pay": 41, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5092", "customer": "Orchard Robotics", "segment": "industrial", "invoice_date": "2026-07-28", "due_date": "2026-08-27", "terms": "Net 30", "amount": 57900, "customer_days_to_pay": 41, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5093", "customer": "Camas Instrument Works", "segment": "aerospace", "invoice_date": "2026-07-29", "due_date": "2026-08-28", "terms": "Net 30", "amount": 33600, "customer_days_to_pay": 40, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5094", "customer": "Camas Instrument Works", "segment": "aerospace", "invoice_date": "2026-08-02", "due_date": "2026-09-01", "terms": "Net 30", "amount": 69300, "customer_days_to_pay": 40, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5095", "customer": "Camas Instrument Works", "segment": "aerospace", "invoice_date": "2026-07-30", "due_date": "2026-08-29", "terms": "Net 30", "amount": 47100, "customer_days_to_pay": 40, "customer_pay_sigma": 6.0, "memo": ""},
{"invoice_id": "INV-5096", "customer": "Falcon Gear Works", "segment": "industrial", "invoice_date": "2026-07-31", "due_date": "2026-08-30", "terms": "Net 30", "amount": 99400, "customer_days_to_pay": 39, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5097", "customer": "Falcon Gear Works", "segment": "industrial", "invoice_date": "2026-08-03", "due_date": "2026-09-02", "terms": "Net 30", "amount": 236400, "customer_days_to_pay": 39, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5098", "customer": "Falcon Gear Works", "segment": "industrial", "invoice_date": "2026-07-30", "due_date": "2026-08-29", "terms": "Net 30", "amount": 134200, "customer_days_to_pay": 39, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5099", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-02", "due_date": "2026-09-01", "terms": "Net 30", "amount": 122500, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5100", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-03", "due_date": "2026-09-02", "terms": "Net 30", "amount": 184300, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5101", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-01", "due_date": "2026-08-31", "terms": "Net 30", "amount": 159000, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5102", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-03", "due_date": "2026-09-02", "terms": "Net 30", "amount": 132300, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5103", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-04", "due_date": "2026-09-03", "terms": "Net 30", "amount": 106900, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5104", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-27", "due_date": "2026-08-26", "terms": "Net 30", "amount": 181300, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5105", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-27", "due_date": "2026-08-26", "terms": "Net 30", "amount": 104400, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5106", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-25", "due_date": "2026-08-24", "terms": "Net 30", "amount": 188600, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5107", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-08-03", "due_date": "2026-09-02", "terms": "Net 30", "amount": 47900, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5108", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-24", "due_date": "2026-08-23", "terms": "Net 30", "amount": 107400, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5109", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-27", "due_date": "2026-08-26", "terms": "Net 30", "amount": 122200, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5110", "customer": "Halvorsen Industrial", "segment": "industrial", "invoice_date": "2026-07-28", "due_date": "2026-08-27", "terms": "Net 30", "amount": 193200, "customer_days_to_pay": 38, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5111", "customer": "Keystone Hydraulics", "segment": "industrial", "invoice_date": "2026-07-29", "due_date": "2026-08-28", "terms": "Net 30", "amount": 177900, "customer_days_to_pay": 37, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5112", "customer": "Keystone Hydraulics", "segment": "industrial", "invoice_date": "2026-07-26", "due_date": "2026-08-25", "terms": "Net 30", "amount": 198900, "customer_days_to_pay": 37, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5113", "customer": "Keystone Hydraulics", "segment": "industrial", "invoice_date": "2026-07-29", "due_date": "2026-08-28", "terms": "Net 30", "amount": 136300, "customer_days_to_pay": 37, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5114", "customer": "Keystone Hydraulics", "segment": "industrial", "invoice_date": "2026-07-22", "due_date": "2026-08-21", "terms": "Net 30", "amount": 87500, "customer_days_to_pay": 37, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5115", "customer": "Keystone Hydraulics", "segment": "industrial", "invoice_date": "2026-07-22", "due_date": "2026-08-21", "terms": "Net 30", "amount": 139400, "customer_days_to_pay": 37, "customer_pay_sigma": 5.0, "memo": ""},
{"invoice_id": "INV-5116", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-22", "due_date": "2026-08-21", "terms": "Net 30", "amount": 62200, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5117", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-21", "due_date": "2026-08-20", "terms": "Net 30", "amount": 153500, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5118", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-20", "due_date": "2026-08-19", "terms": "Net 30", "amount": 130400, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5119", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-22", "due_date": "2026-08-21", "terms": "Net 30", "amount": 151800, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5120", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-21", "due_date": "2026-08-20", "terms": "Net 30", "amount": 163800, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5121", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-22", "due_date": "2026-08-21", "terms": "Net 30", "amount": 98300, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""},
{"invoice_id": "INV-5122", "customer": "Redline Motion Systems", "segment": "industrial", "invoice_date": "2026-07-19", "due_date": "2026-08-18", "terms": "Net 30", "amount": 220000, "customer_days_to_pay": 36, "customer_pay_sigma": 4.0, "memo": ""}
]


In [ ]:
%%writefile data/ap_open_bills.json
[
{"bill_id": "AP-9001", "vendor": "Rainier Tool & Carbide", "category": "tooling", "due_date": "2026-09-28", "amount": 120000, "discretionary": true, "memo": "Tooling package upgrade, deferrable per ops"},
{"bill_id": "AP-9002", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-10-08", "amount": 85000, "discretionary": true, "memo": "Spindle rebuild on backup mill, non-critical"},
{"bill_id": "AP-9003", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-09-21", "amount": 45000, "discretionary": true, "memo": "Packaging inventory top-up, above reorder point"},
{"bill_id": "AP-9004", "vendor": "CNC Spares Direct", "category": "maintenance", "due_date": "2026-10-15", "amount": 60000, "discretionary": true, "memo": "Spare parts stocking order, deferrable"},
{"bill_id": "AP-9005", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-09-02", "amount": 140800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9006", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-09-03", "amount": 148800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9007", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-10-15", "amount": 135400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9008", "vendor": "Columbia Heat Treat", "category": "outside processing", "due_date": "2026-09-10", "amount": 39000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9009", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-09-25", "amount": 164100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9010", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-09-14", "amount": 146700, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9011", "vendor": "CNC Spares Direct", "category": "maintenance", "due_date": "2026-09-15", "amount": 158500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9012", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-08-27", "amount": 103100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9013", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-10-07", "amount": 164500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9014", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-09-29", "amount": 28600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9015", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-08-25", "amount": 26100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9016", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-10-02", "amount": 24500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9017", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-09-10", "amount": 60800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9018", "vendor": "Evergreen Industrial Gases", "category": "consumables", "due_date": "2026-10-01", "amount": 60200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9019", "vendor": "CNC Spares Direct", "category": "maintenance", "due_date": "2026-09-04", "amount": 50600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9020", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-07", "amount": 110000, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9021", "vendor": "Tacoma Abrasives", "category": "consumables", "due_date": "2026-09-30", "amount": 27400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9022", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-10-15", "amount": 113600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9023", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-10-01", "amount": 47300, "discretionary": false, "memo": ""},
{"bill_id": "AP-9024", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-09-21", "amount": 127300, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9025", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-09-30", "amount": 24600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9026", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-09-29", "amount": 69900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9027", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-09-15", "amount": 168000, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9028", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-08-25", "amount": 105500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9029", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-09-21", "amount": 148200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9030", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-09-21", "amount": 124200, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9031", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-09-07", "amount": 116800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9032", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-10-13", "amount": 51200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9033", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-08-31", "amount": 111100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9034", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-10-12", "amount": 27500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9035", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-02", "amount": 146600, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9036", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-10-13", "amount": 171400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9037", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-08-27", "amount": 154800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9038", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-08-31", "amount": 29200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9039", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-09-25", "amount": 74300, "discretionary": false, "memo": ""},
{"bill_id": "AP-9040", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-10-02", "amount": 71000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9041", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-08", "amount": 38900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9042", "vendor": "Rainier Tool & Carbide", "category": "tooling", "due_date": "2026-10-05", "amount": 119300, "discretionary": false, "memo": ""},
{"bill_id": "AP-9043", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-09-14", "amount": 146000, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9044", "vendor": "Olympic Alloys", "category": "raw material", "due_date": "2026-09-08", "amount": 70400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9045", "vendor": "Columbia Heat Treat", "category": "outside processing", "due_date": "2026-09-29", "amount": 156200, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9046", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-08-28", "amount": 145900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9047", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-10-06", "amount": 41500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9048", "vendor": "CNC Spares Direct", "category": "maintenance", "due_date": "2026-09-07", "amount": 141200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9049", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-09-09", "amount": 159300, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9050", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-08-28", "amount": 52200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9051", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-08-31", "amount": 111000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9052", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-08", "amount": 121200, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9053", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-09-07", "amount": 116600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9054", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-09-08", "amount": 36400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9055", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-09-11", "amount": 124700, "discretionary": false, "memo": ""},
{"bill_id": "AP-9056", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-09-30", "amount": 120100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9057", "vendor": "Evergreen Industrial Gases", "category": "consumables", "due_date": "2026-09-18", "amount": 150100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9058", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-10-13", "amount": 146900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9059", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-09-01", "amount": 72500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9060", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-10-16", "amount": 134200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9061", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-09-10", "amount": 156900, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9062", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-09-03", "amount": 160900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9063", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-09-23", "amount": 46600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9064", "vendor": "CNC Spares Direct", "category": "maintenance", "due_date": "2026-08-27", "amount": 25500, "discretionary": false, "memo": ""},
{"bill_id": "AP-9065", "vendor": "Columbia Heat Treat", "category": "outside processing", "due_date": "2026-09-25", "amount": 123100, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9066", "vendor": "Olympic Alloys", "category": "raw material", "due_date": "2026-09-18", "amount": 54900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9067", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-10-05", "amount": 109400, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9068", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-10-09", "amount": 169000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9069", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-09-29", "amount": 80600, "discretionary": false, "memo": ""},
{"bill_id": "AP-9070", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-09-25", "amount": 60800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9071", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-09-15", "amount": 92700, "discretionary": false, "memo": ""},
{"bill_id": "AP-9072", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-01", "amount": 124100, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9073", "vendor": "Cascadia Titanium Supply", "category": "raw material", "due_date": "2026-10-09", "amount": 37100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9074", "vendor": "Evergreen Industrial Gases", "category": "consumables", "due_date": "2026-09-11", "amount": 80800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9075", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-08-25", "amount": 42200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9076", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-08-24", "amount": 124900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9077", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-09-23", "amount": 151200, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9078", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-09-16", "amount": 80200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9079", "vendor": "Baker Calibration Services", "category": "maintenance", "due_date": "2026-10-02", "amount": 79400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9080", "vendor": "Tacoma Abrasives", "category": "consumables", "due_date": "2026-10-13", "amount": 105700, "discretionary": false, "memo": ""},
{"bill_id": "AP-9081", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-09-14", "amount": 54900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9082", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-10-08", "amount": 60000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9083", "vendor": "Willamette Plating & Anodize", "category": "outside processing", "due_date": "2026-09-23", "amount": 72900, "discretionary": false, "memo": ""},
{"bill_id": "AP-9084", "vendor": "Rainier Tool & Carbide", "category": "tooling", "due_date": "2026-09-16", "amount": 92800, "discretionary": false, "memo": ""},
{"bill_id": "AP-9085", "vendor": "Skagit Machine Rebuild", "category": "maintenance", "due_date": "2026-08-27", "amount": 34100, "discretionary": false, "memo": ""},
{"bill_id": "AP-9086", "vendor": "Harbor Packaging", "category": "consumables", "due_date": "2026-08-27", "amount": 139000, "discretionary": false, "memo": ""},
{"bill_id": "AP-9087", "vendor": "Sound Fastener Supply", "category": "raw material", "due_date": "2026-09-16", "amount": 111800, "discretionary": false, "memo": "Aerospace ramp material buy"},
{"bill_id": "AP-9088", "vendor": "Kootenai Coatings", "category": "outside processing", "due_date": "2026-09-30", "amount": 68200, "discretionary": false, "memo": ""},
{"bill_id": "AP-9089", "vendor": "Puget Freight Lines", "category": "freight", "due_date": "2026-09-25", "amount": 33400, "discretionary": false, "memo": ""},
{"bill_id": "AP-9090", "vendor": "Selkirk Specialty Steel", "category": "raw material", "due_date": "2026-09-23", "amount": 140700, "discretionary": false, "memo": "Aerospace ramp material buy"}
]


In [ ]:
%%writefile data/fixed_schedule.csv
week,date,item,category,amount,discretionary
1,2026-08-25,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,350000,False
1,2026-08-26,"Health benefits premium, monthly",benefits,205000,False
1,2026-08-27,Utilities & plant services,operations,70000,False
1,2026-08-28,Biweekly payroll,payroll,745000,False
2,2026-08-31,"Facility rent, monthly",occupancy,95000,False
2,2026-09-01,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,420000,False
2,2026-09-03,Utilities & plant services,operations,70000,False
3,2026-09-08,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,500000,False
3,2026-09-10,Utilities & plant services,operations,70000,False
3,2026-09-11,Biweekly payroll,payroll,745000,False
4,2026-09-15,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,600000,False
4,2026-09-17,Utilities & plant services,operations,70000,False
5,2026-09-22,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,700000,False
5,2026-09-23,"Health benefits premium, monthly",benefits,205000,False
5,2026-09-23,Semiannual property & casualty insurance premium,insurance,450000,False
5,2026-09-24,Utilities & plant services,operations,70000,False
5,2026-09-25,"Biweekly payroll (incl. second shift, aerospace ramp)",payroll,815000,False
6,2026-09-28,"Facility rent, monthly",occupancy,95000,False
6,2026-09-29,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,780000,False
6,2026-09-30,Capex deposit: Hermle C42 5-axis machining center,capex,800000,True
6,2026-10-01,Utilities & plant services,operations,70000,False
7,2026-10-06,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,860000,False
7,2026-10-08,Utilities & plant services,operations,70000,False
7,2026-10-09,"Biweekly payroll (incl. second shift, aerospace ramp)",payroll,815000,False
8,2026-10-13,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,1290000,False
8,2026-10-15,Utilities & plant services,operations,70000,False
9,2026-10-20,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,1350000,False
9,2026-10-21,"Health benefits premium, monthly",benefits,205000,False
9,2026-10-21,Seller note earn-out payment (Granite Peak acquisition),acquisition,1000000,False
9,2026-10-22,Utilities & plant services,operations,70000,False
9,2026-10-23,"Biweekly payroll (incl. second shift, aerospace ramp)",payroll,815000,False
10,2026-10-26,"Facility rent, monthly",occupancy,95000,False
10,2026-10-27,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,1100000,False
10,2026-10-28,Estimated federal income tax payment,tax,300000,False
10,2026-10-29,Utilities & plant services,operations,70000,False
11,2026-11-03,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,1300000,False
11,2026-11-05,Utilities & plant services,operations,70000,False
11,2026-11-06,"Biweekly payroll (incl. second shift, aerospace ramp)",payroll,815000,False
12,2026-11-10,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,1420000,False
12,2026-11-12,Utilities & plant services,operations,70000,False
13,2026-11-17,"Material purchases, forecast (aerospace ramp POs, Net 30)",materials forecast,770000,False
13,2026-11-19,Utilities & plant services,operations,70000,False
13,2026-11-20,"Biweekly payroll (incl. second shift, aerospace ramp)",payroll,815000,False
13,2026-11-20,Term loan quarterly amortization,debt service,1250000,False
13,2026-11-20,Term loan quarterly interest,debt service,950000,False


In [ ]:
%%writefile data/sales_forecast.csv
week,week_start,billings,sigma,base_lag_days,vantage_share,vantage_lag_days
1,2026-08-24,1380000,138000,61,0.22,64
2,2026-08-31,1420000,142000,61,0.22,64
3,2026-09-07,1460000,161000,61,0.22,64
4,2026-09-14,1520000,167000,61,0.22,64
5,2026-09-21,1580000,190000,61,0.22,64
6,2026-09-28,1660000,199000,61,0.22,64
7,2026-10-05,1760000,229000,61,0.22,64
8,2026-10-12,1880000,244000,61,0.22,64
9,2026-10-19,1990000,279000,61,0.22,64
10,2026-10-26,2080000,291000,61,0.22,64
11,2026-11-02,2150000,322000,61,0.22,64
12,2026-11-09,2200000,330000,61,0.22,64
13,2026-11-16,2240000,336000,61,0.22,64


In [ ]:
%%writefile data/historical_actuals.csv
week_ending,forecast_collections,forecast_vantage,actual_collections,actual_vantage,actual_disbursements,actual_ending_cash
2026-07-31,1950000,310000,1905000,305000,1985000,3096000
2026-08-07,1880000,420000,1452000,20000,1860000,2688000
2026-08-14,1760000,180000,1741000,175000,2015000,2414000
2026-08-21,1850000,240000,1896000,262000,1910000,2400000


## 1.5 · `logger.py` — the audit trail

Small on purpose. Every pipeline event — every tool call, its inputs, its outputs, the
agent's narration, the reviewer's decisions — lands in an append-only list with a UTC
timestamp, then persists to `output/audit_log.json` and `output/replay_capture.json`
(the replay file is the live-demo safety net for Phase 2). When someone asks "where did
this number come from" six weeks later, this file is the answer.

In [ ]:
%%writefile cashflow_harness/logger.py
import json
from datetime import datetime, timezone

from .config import OUTPUT_DIR


class AuditLogger:
    """Append-only audit trail. Every pipeline event lands here with a
    timestamp; write() persists the trail and write_replay() persists the
    same sequence for the Phase 2 replay safety net."""

    def __init__(self):
        self.events: list[dict] = []

    def log(self, event_type: str, data: dict) -> None:
        self.events.append({
            "event": event_type,
            "data": data,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

    def write(self) -> str:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        path = OUTPUT_DIR / "audit_log.json"
        with open(path, "w") as f:
            json.dump({"events": self.events, "event_count": len(self.events)}, f, indent=2)
        return str(path)

    def write_replay(self) -> str:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        path = OUTPUT_DIR / "replay_capture.json"
        with open(path, "w") as f:
            json.dump({"events": self.events}, f, indent=2)
        return str(path)


## 1.6 · `tools.py` — the agent's tool belt

Ten tools in three patterns:

- **Data readers** (1–3): load the facility, the receivables (with per-customer behavior
  stats and term-stretch flags), and the payables/fixed schedule. They return structured
  summaries, not raw dumps — the tool shapes what the model attends to.
- **Computation tools** (4–7): the deterministic forecast, the Monte Carlo, the scenario
  engine, and the trailing variance bridge. `run_scenario` supports single what-ifs
  (slip a customer, accelerate a collection, defer an item, flex sales, stretch AP) and
  a `driver_sweep` that ranks what moves week-13 headroom most — the tornado.
- **Pass-through and governance tools** (8–10): `draft_cash_narrative` forces the
  narrative into a validated structure (so it lands in the audit trail, not just chat
  text), `submit_for_review` is the mandatory human gate, and `log_output` writes the
  final record.

A finance subtlety the scenario engine makes teachable: covenant liquidity
(cash + availability) is *invariant* to revolver draws and sweeps, so **only flows that
cross the week-13 boundary move deterministic headroom**. Deferring the capex deposit
two weeks does nothing; deferring it past the test date, or pulling Vantage collections
back inside the quarter, moves real money. Watch the agent discover this in Part 2.

In [ ]:
%%writefile cashflow_harness/tools.py
"""Tool implementations for the Cascade Precision cash flow agent.

Every tool is a plain, importable Python function. The Phase 1 notebook and
the Phase 2 FastAPI server call these exact functions; if a number ever
differs between the two, the wrapper is wrong, not the engine.

Data-reading tools return structured results. Computation tools call the
deterministic engine and the Monte Carlo layer. Pass-through tools
(draft_cash_narrative) validate the agent's structured input and return a
confirmation so the payload is captured for the audit log.
"""

import copy
import csv
import json
from collections import defaultdict
from datetime import date, timedelta

from . import engine, montecarlo
from .config import DATA_DIR, MC_ITERATIONS, MC_SEED, OUTPUT_DIR

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------


def _load_facility() -> dict:
    with open(DATA_DIR / "facility_terms.json") as f:
        return json.load(f)


def _load_invoices() -> list[dict]:
    with open(DATA_DIR / "ar_open_invoices.json") as f:
        return json.load(f)


def _load_bills() -> list[dict]:
    with open(DATA_DIR / "ap_open_bills.json") as f:
        return json.load(f)


def _load_fixed() -> list[dict]:
    with open(DATA_DIR / "fixed_schedule.csv") as f:
        return [
            dict(r, week=int(r["week"]), amount=float(r["amount"]),
                 discretionary=r["discretionary"] == "True")
            for r in csv.DictReader(f)
        ]


def _load_sales() -> list[dict]:
    with open(DATA_DIR / "sales_forecast.csv") as f:
        return [
            dict(r, week=int(r["week"]), billings=float(r["billings"]),
                 sigma=float(r["sigma"]), base_lag_days=float(r["base_lag_days"]),
                 vantage_share=float(r["vantage_share"]),
                 vantage_lag_days=float(r["vantage_lag_days"]))
            for r in csv.DictReader(f)
        ]


def _load_history() -> list[dict]:
    with open(DATA_DIR / "historical_actuals.csv") as f:
        return [
            {k: (v if k == "week_ending" else float(v)) for k, v in r.items()}
            for r in csv.DictReader(f)
        ]


def _week1(facility: dict) -> date:
    return date.fromisoformat(facility["week1_monday"])


def _run_deterministic(facility, invoices, bills, fixed, sales) -> dict:
    receipts, disb = engine.build_weekly_flows(
        invoices, bills, fixed, sales, _week1(facility), facility["num_weeks"]
    )
    return engine.run_forecast(
        beginning_cash=facility["beginning_cash"],
        revolver_limit=facility["revolver_limit"],
        revolver_drawn=facility["revolver_drawn"],
        operating_floor=facility["operating_cash_floor"],
        covenant_threshold=facility["covenant"]["threshold"],
        weekly_receipts=receipts,
        weekly_disbursements=disb,
        sweep_buffer=facility["sweep_buffer"],
        draw_increment=facility["draw_increment"],
        covenant_test_week=facility["covenant"]["test_week"],
    )


# ---------------------------------------------------------------------------
# Tools 1-3: load the position
# ---------------------------------------------------------------------------


def load_facility_and_position() -> dict:
    facility = _load_facility()
    availability = facility["revolver_limit"] - facility["revolver_drawn"]
    return {
        "company": facility["company"],
        "as_of_close": facility["as_of_close"],
        "week1_monday": facility["week1_monday"],
        "horizon_weeks": facility["num_weeks"],
        "beginning_cash": facility["beginning_cash"],
        "revolver": {
            "limit": facility["revolver_limit"],
            "drawn": facility["revolver_drawn"],
            "availability": availability,
        },
        "beginning_liquidity": facility["beginning_cash"] + availability,
        "operating_cash_floor": facility["operating_cash_floor"],
        "sweep_buffer": facility["sweep_buffer"],
        "draw_increment": facility["draw_increment"],
        "covenant": facility["covenant"],
        "term_loan": facility["term_loan"],
    }


def load_receivables() -> dict:
    facility = _load_facility()
    invoices = _load_invoices()
    week1 = _week1(facility)
    total = sum(i["amount"] for i in invoices)

    by_customer: dict[str, dict] = {}
    for inv in invoices:
        c = by_customer.setdefault(inv["customer"], {
            "customer": inv["customer"],
            "segment": inv["segment"],
            "terms": inv["terms"],
            "days_to_pay": inv["customer_days_to_pay"],
            "pay_sigma": inv["customer_pay_sigma"],
            "open_ar": 0,
            "invoice_count": 0,
        })
        c["open_ar"] += inv["amount"]
        c["invoice_count"] += 1

    customers = sorted(by_customer.values(), key=lambda c: -c["open_ar"])
    for c in customers:
        c["share_of_ar"] = round(c["open_ar"] / total, 4)
        terms_days = int(c["terms"].split()[-1])
        c["stretch_days_vs_terms"] = round(c["days_to_pay"] - terms_days, 1)
        c["term_stretch_flag"] = c["stretch_days_vs_terms"] >= 15

    expected_by_week = defaultdict(float)
    for inv in invoices:
        w = engine.date_to_week(engine.expected_pay_date(inv), week1, facility["num_weeks"])
        expected_by_week["beyond_horizon" if w > facility["num_weeks"] else f"week_{w}"] += inv["amount"]

    top_invoices = sorted(invoices, key=lambda i: -i["amount"])[:8]
    return {
        "total_open_ar": total,
        "invoice_count": len(invoices),
        "customers": customers,
        "top5_concentration": round(sum(c["open_ar"] for c in customers[:5]) / total, 4),
        "flagged_term_stretch": [c["customer"] for c in customers if c["term_stretch_flag"]],
        "largest_open_invoices": [
            {k: inv[k] for k in ("invoice_id", "customer", "amount", "invoice_date",
                                 "due_date", "terms", "memo")}
            for inv in top_invoices
        ],
        "expected_collections_by_week": {
            k: round(v) for k, v in sorted(expected_by_week.items())
        },
        "note": "Expected collection weeks use observed customer payment behavior "
                "(days to pay), not stated terms.",
    }


def load_payables_and_fixed() -> dict:
    facility = _load_facility()
    bills = _load_bills()
    fixed = _load_fixed()
    week1 = _week1(facility)
    num_weeks = facility["num_weeks"]

    ap_by_week = defaultdict(float)
    for b in bills:
        w = engine.date_to_week(date.fromisoformat(b["due_date"]), week1, num_weeks)
        if w <= num_weeks:
            ap_by_week[w] += b["amount"]

    fixed_by_week_cat = defaultdict(lambda: defaultdict(float))
    for r in fixed:
        fixed_by_week_cat[r["week"]][r["category"]] += r["amount"]

    weekly = []
    for w in range(1, num_weeks + 1):
        cats = dict(fixed_by_week_cat.get(w, {}))
        row = {"week": w, "ap_due": round(ap_by_week.get(w, 0))}
        row.update({k.replace(" ", "_"): round(v) for k, v in sorted(cats.items())})
        row["total"] = round(ap_by_week.get(w, 0) + sum(cats.values()))
        weekly.append(row)

    discretionary = [
        {"source": "fixed_schedule", "week": r["week"], "item": r["item"], "amount": r["amount"]}
        for r in fixed if r["discretionary"]
    ] + [
        {"source": "open_ap", "week": engine.date_to_week(date.fromisoformat(b["due_date"]), week1, num_weeks),
         "item": f'{b["vendor"]}: {b["memo"] or b["category"]}', "amount": b["amount"]}
        for b in bills if b["discretionary"]
    ]

    one_timers = [
        {"week": r["week"], "item": r["item"], "category": r["category"], "amount": r["amount"]}
        for r in fixed
        if r["category"] in ("insurance", "capex", "acquisition", "tax", "debt service")
    ]

    return {
        "total_open_ap": sum(b["amount"] for b in bills),
        "ap_bill_count": len(bills),
        "weekly_disbursement_schedule": weekly,
        "one_time_items": one_timers,
        "discretionary_items": discretionary,
        "note": "AP is heaviest in weeks 3-7 from aerospace ramp material buys. "
                "Forecast material purchases (Net 30 on new ramp POs) continue "
                "through week 13.",
    }


# ---------------------------------------------------------------------------
# Tools 4-5: the engines
# ---------------------------------------------------------------------------


def build_deterministic_forecast() -> dict:
    facility = _load_facility()
    result = _run_deterministic(facility, _load_invoices(), _load_bills(), _load_fixed(), _load_sales())
    result["note"] = (
        "Single-point roll-forward using observed customer payment behavior. "
        "Revolver auto-draws in increments to defend the operating floor; "
        "excess cash above floor + buffer sweeps the revolver down."
    )
    return result


def run_monte_carlo(iterations: int | None = None, seed: int | None = None) -> dict:
    facility = _load_facility()
    return montecarlo.run_monte_carlo(
        _load_invoices(), _load_bills(), _load_fixed(), _load_sales(), facility,
        iterations=iterations or MC_ITERATIONS,
        seed=seed if seed is not None else MC_SEED,
    )


# ---------------------------------------------------------------------------
# Tool 6: scenarios
# ---------------------------------------------------------------------------


def _apply_scenario(scenario: dict, invoices, bills, fixed, sales) -> tuple[list, list, list, list]:
    """Return transformed copies of the four flow inputs for one scenario."""
    invoices = copy.deepcopy(invoices)
    bills = copy.deepcopy(bills)
    fixed = copy.deepcopy(fixed)
    sales = copy.deepcopy(sales)
    stype = scenario["scenario_type"]

    if stype == "slip_collections":
        customer = scenario.get("customer")
        days = float(scenario["days"])
        for inv in invoices:
            if customer is None or inv["customer"] == customer:
                inv["customer_days_to_pay"] += days
        for row in sales:
            if customer is None:
                row["base_lag_days"] += days
                row["vantage_lag_days"] += days
            elif customer == "Vantage Aerospace":
                row["vantage_lag_days"] += days

    elif stype == "accelerate_collection":
        customer = scenario["customer"]
        to_days = float(scenario["days"])
        for inv in invoices:
            if inv["customer"] == customer:
                inv["customer_days_to_pay"] = to_days
                inv["customer_pay_sigma"] = min(inv["customer_pay_sigma"], 3.0)
        if customer == "Vantage Aerospace":
            for row in sales:
                row["vantage_lag_days"] = to_days

    elif stype == "defer_item":
        query = scenario["item"].lower()
        to_week = int(scenario["to_week"])
        facility = _load_facility()
        week1 = _week1(facility)
        moved = False
        for r in fixed:
            if query in r["item"].lower():
                r["week"] = to_week
                r["date"] = (week1 + timedelta(days=(to_week - 1) * 7 + 2)).isoformat()
                moved = True
        for b in bills:
            if query in b["vendor"].lower() or query in (b.get("memo") or "").lower():
                b["due_date"] = (week1 + timedelta(days=(to_week - 1) * 7 + 2)).isoformat()
                moved = True
        if not moved:
            raise ValueError(f"No fixed-schedule item or bill matches '{scenario['item']}'")

    elif stype == "sales_change":
        factor = 1 + float(scenario["pct"]) / 100
        for row in sales:
            row["billings"] *= factor
            row["sigma"] *= factor

    elif stype == "stretch_ap":
        days = int(scenario["days"])
        for b in bills:
            b["due_date"] = (date.fromisoformat(b["due_date"]) + timedelta(days=days)).isoformat()

    else:
        raise ValueError(f"Unknown scenario_type: {stype}")

    return invoices, bills, fixed, sales


def _evaluate(facility, invoices, bills, fixed, sales, mc_seed=MC_SEED) -> dict:
    det = _run_deterministic(facility, invoices, bills, fixed, sales)
    mc = montecarlo.run_monte_carlo(
        invoices, bills, fixed, sales, facility, iterations=MC_ITERATIONS, seed=mc_seed
    )
    return {
        "trough_week": det["trough_week"],
        "trough_cash": det["trough_cash"],
        "test_week_liquidity": det["test_week_liquidity"],
        "covenant_headroom": det["covenant_headroom"],
        "p_covenant_breach": mc["p_covenant_breach"],
    }


DRIVER_SWEEP = [
    {"label": "Vantage pays 10 days later", "scenario_type": "slip_collections",
     "customer": "Vantage Aerospace", "days": 10},
    {"label": "Vantage accelerated to Net 45 terms", "scenario_type": "accelerate_collection",
     "customer": "Vantage Aerospace", "days": 45},
    {"label": "All customers pay 7 days later", "scenario_type": "slip_collections", "days": 7},
    {"label": "Sales 10% below forecast", "scenario_type": "sales_change", "pct": -10},
    {"label": "Sales 10% above forecast", "scenario_type": "sales_change", "pct": 10},
    {"label": "Defer capex deposit 2 weeks (wk 6 to wk 8)", "scenario_type": "defer_item",
     "item": "capex deposit", "to_week": 8},
    {"label": "Defer capex deposit past quarter end (wk 14)", "scenario_type": "defer_item",
     "item": "capex deposit", "to_week": 14},
    {"label": "Stretch all AP 7 days", "scenario_type": "stretch_ap", "days": 7},
]


def run_scenario(scenario_type: str, customer: str | None = None, days: float | None = None,
                 pct: float | None = None, item: str | None = None,
                 to_week: int | None = None) -> dict:
    """Run one what-if, or the full driver sweep when scenario_type='driver_sweep'."""
    facility = _load_facility()
    base_inputs = (_load_invoices(), _load_bills(), _load_fixed(), _load_sales())
    baseline = _evaluate(facility, *base_inputs)

    def run_one(spec: dict) -> dict:
        transformed = _apply_scenario(spec, *base_inputs)
        result = _evaluate(facility, *transformed)
        result["delta_covenant_headroom"] = round(result["covenant_headroom"] - baseline["covenant_headroom"], 2)
        result["delta_trough_cash"] = round(result["trough_cash"] - baseline["trough_cash"], 2)
        result["delta_p_breach"] = round(result["p_covenant_breach"] - baseline["p_covenant_breach"], 4)
        return result

    if scenario_type == "driver_sweep":
        results = []
        for spec in DRIVER_SWEEP:
            r = run_one(spec)
            r["label"] = spec["label"]
            results.append(r)
        results.sort(key=lambda r: -abs(r["delta_covenant_headroom"]))
        return {
            "baseline": baseline,
            "sweep": results,
            "note": "Sweep ranked by absolute impact on week-13 covenant headroom. "
                    "Moves inside the quarter shift the trough and breach odds; only "
                    "flows crossing the week-13 boundary move deterministic headroom.",
        }

    spec = {"scenario_type": scenario_type, "customer": customer, "days": days,
            "pct": pct, "item": item, "to_week": to_week}
    result = run_one(spec)
    return {"baseline": baseline, "scenario": spec, "result": result}


# ---------------------------------------------------------------------------
# Tool 7: trailing variance
# ---------------------------------------------------------------------------


def compute_variance() -> dict:
    history = _load_history()
    fc_total = sum(r["forecast_collections"] for r in history)
    ac_total = sum(r["actual_collections"] for r in history)
    miss = ac_total - fc_total

    vantage_miss = sum(r["actual_vantage"] - r["forecast_vantage"] for r in history)
    other_miss = miss - vantage_miss

    weekly = []
    for r in history:
        weekly.append({
            "week_ending": r["week_ending"],
            "forecast_collections": round(r["forecast_collections"]),
            "actual_collections": round(r["actual_collections"]),
            "variance": round(r["actual_collections"] - r["forecast_collections"]),
            "vantage_variance": round(r["actual_vantage"] - r["forecast_vantage"]),
            "other_variance": round((r["actual_collections"] - r["forecast_collections"])
                                    - (r["actual_vantage"] - r["forecast_vantage"])),
            "actual_disbursements": round(r["actual_disbursements"]),
            "actual_ending_cash": round(r["actual_ending_cash"]),
        })

    return {
        "trailing_weeks": len(history),
        "forecast_collections_total": round(fc_total),
        "actual_collections_total": round(ac_total),
        "total_variance": round(miss),
        "variance_pct_of_forecast": round(miss / fc_total, 4),
        "attribution": {
            "vantage_timing": round(vantage_miss),
            "all_other": round(other_miss),
            "vantage_share_of_miss": round(vantage_miss / miss, 4) if miss else 0,
        },
        "weekly_bridge": weekly,
        "note": "Timing vs volume: the Vantage variance is a slipped invoice that "
                "remains fully collectible (timing). The residual spread across "
                "other customers nets small and reads as normal volume noise.",
    }


# ---------------------------------------------------------------------------
# Tools 8-10: narrative, review gate, audit output
# ---------------------------------------------------------------------------


def draft_cash_narrative(sections: list[dict], recommended_action: dict) -> dict:
    """Pass-through: validate and confirm storage of the structured narrative."""
    required = {"title", "content", "confidence"}
    for s in sections:
        missing = required - set(s)
        if missing:
            raise ValueError(f"Narrative section missing fields: {missing}")
    if not {"action", "rationale"} <= set(recommended_action):
        raise ValueError("recommended_action needs 'action' and 'rationale'")
    return {
        "status": "stored",
        "section_count": len(sections),
        "section_titles": [s["title"] for s in sections],
        "recommended_action": recommended_action.get("action", ""),
    }


def submit_for_review(message: str) -> dict:
    # CLI/default mode auto-approves. The agent loop intercepts this tool and
    # routes it through the review-gate callback (notebook prompt or Phase 2
    # WebSocket gate) when one is provided.
    return {
        "status": "auto_approved",
        "message": "No review handler attached: all sections auto-approved.",
    }


def log_output(final_sections: list[dict], recommended_action_disposition: dict,
               summary: str) -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # final_report.json, not audit_log.json: the AuditLogger owns audit_log.json
    # and would overwrite this at the end of the run.
    path = OUTPUT_DIR / "final_report.json"
    with open(path, "w") as f:
        json.dump({
            "summary": summary,
            "final_sections": final_sections,
            "recommended_action_disposition": recommended_action_disposition,
            "section_count": len(final_sections),
        }, f, indent=2)
    return {"status": "written", "path": str(path), "entry_count": len(final_sections)}


TOOL_FUNCTIONS = {
    "load_facility_and_position": load_facility_and_position,
    "load_receivables": load_receivables,
    "load_payables_and_fixed": load_payables_and_fixed,
    "build_deterministic_forecast": build_deterministic_forecast,
    "run_monte_carlo": run_monte_carlo,
    "run_scenario": run_scenario,
    "compute_variance": compute_variance,
    "draft_cash_narrative": draft_cash_narrative,
    "submit_for_review": submit_for_review,
    "log_output": log_output,
}


def execute_tool(tool_name: str, tool_input: dict) -> dict:
    fn = TOOL_FUNCTIONS.get(tool_name)
    if fn is None:
        return {"error": f"Unknown tool: {tool_name}"}
    try:
        return fn(**tool_input)
    except (TypeError, ValueError, KeyError) as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}


## 1.7 · `tool_schemas.py` — the contract with the model

The Anthropic tool definitions, one per function in `tools.py`. Descriptions carry the
domain intent ("the number that changes the meeting"), and the pass-through schemas force
structure: the narrative must arrive as sections with confidence levels and open flags,
plus a single recommended action with a rationale and quantified effect. Structured
output isn't cosmetic — it is what makes the review gate and the audit trail possible.

In [ ]:
%%writefile cashflow_harness/tool_schemas.py
"""Anthropic tool definitions for the cash flow agent.

Names and parameters mirror the functions in tools.py one for one. The
pass-through tools (draft_cash_narrative) force structured output so the
narrative is captured in the audit trail rather than living only in
free-form model text.
"""

TOOLS = [
    {
        "name": "load_facility_and_position",
        "description": "Load the opening treasury position and facility terms: beginning cash, revolver limit / drawn / availability, the operating cash floor and sweep policy, the liquidity covenant (threshold, test week, consequence), and the term loan debt service schedule.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "load_receivables",
        "description": "Load open accounts receivable: per-customer subtotals with share of AR, stated terms versus observed days-to-pay (with sigma), concentration, the largest open invoices, and expected collections by week based on observed payment behavior. Customers whose payment behavior has stretched well past terms are flagged.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "load_payables_and_fixed",
        "description": "Load scheduled disbursements: open AP bucketed by due week, plus the fixed schedule (payroll, rent, benefits, utilities, one-time items, debt service, forecast ramp material purchases) by week and category. Returns one-time items and the discretionary list separately.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "build_deterministic_forecast",
        "description": "Run the deterministic 13-week cash roll-forward with revolver mechanics (auto-draw to the floor, sweep above floor + buffer). Returns the weekly table (receipts, disbursements, pre-revolver cash, draws/paydowns, revolver balance, ending cash, covenant liquidity), the trough week and amount, and week-13 covenant headroom.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "run_monte_carlo",
        "description": "Run the probabilistic layer: resample every invoice's pay timing from the customer's observed behavior and every week's new billings from the forecast sigma, then re-run the deterministic engine per iteration. Returns P10/P50/P90 ending cash and liquidity by week, the trough distribution, week-13 liquidity percentiles, the probability the floor cannot be restored within the revolver limit, and the probability of a week-13 covenant breach. Seeded and reproducible.",
        "input_schema": {
            "type": "object",
            "properties": {
                "iterations": {"type": "integer", "description": "Number of iterations (default 1000)."},
                "seed": {"type": "integer", "description": "Random seed (default 42, the reproducible demo seed)."},
            },
            "required": [],
        },
    },
    {
        "name": "run_scenario",
        "description": "Run a what-if against the base forecast and report the new trough, week-13 headroom, and breach probability with deltas. Use scenario_type 'driver_sweep' (no other arguments) to run the standard tornado sweep ranking what moves week-13 covenant headroom most. Other types: 'slip_collections' (customer optional, days required: everyone or one customer pays N days later), 'accelerate_collection' (customer and days required: pin a customer's collections to N days, e.g. Vantage back to 45-day terms), 'defer_item' (item and to_week required: move a fixed-schedule item or discretionary bill to a later week; week 14 pushes it past the covenant test), 'sales_change' (pct required: flex forecast billings), 'stretch_ap' (days required: pay all suppliers N days later).",
        "input_schema": {
            "type": "object",
            "properties": {
                "scenario_type": {
                    "type": "string",
                    "enum": ["driver_sweep", "slip_collections", "accelerate_collection",
                             "defer_item", "sales_change", "stretch_ap"],
                },
                "customer": {"type": "string", "description": "Customer name, e.g. 'Vantage Aerospace'."},
                "days": {"type": "number", "description": "Days to slip, stretch, or pin to."},
                "pct": {"type": "number", "description": "Percent change to forecast billings, e.g. -10."},
                "item": {"type": "string", "description": "Substring matching a fixed-schedule item or vendor, e.g. 'capex deposit'."},
                "to_week": {"type": "integer", "description": "Destination week for defer_item (14 = past quarter end)."},
            },
            "required": ["scenario_type"],
        },
    },
    {
        "name": "compute_variance",
        "description": "Bridge the trailing 4 weeks of actual collections against the prior forecast. Attributes the miss between the flagged customer's slipped timing and all-other volume noise, with a weekly bridge table.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "draft_cash_narrative",
        "description": "Submit the structured treasury narrative for review. Sections must cover: (a) position and covenant summary, (b) the 13-week path and the trough, (c) what the probabilistic view adds and the breach risk, (d) recommended actions with their effect on covenant headroom. Each section carries a confidence level and open flags. Also submit the single recommended treasury action with its rationale and quantified effect.",
        "input_schema": {
            "type": "object",
            "properties": {
                "sections": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "content": {"type": "string"},
                            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                            "open_flags": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "Anything unresolved or that the reviewer should verify.",
                            },
                        },
                        "required": ["title", "content", "confidence"],
                    },
                },
                "recommended_action": {
                    "type": "object",
                    "properties": {
                        "action": {"type": "string", "description": "The single recommended treasury action."},
                        "rationale": {"type": "string"},
                        "expected_headroom_effect": {"type": "number", "description": "Dollar change to week-13 covenant headroom."},
                        "expected_breach_prob_effect": {"type": "number", "description": "Change to breach probability, e.g. -0.12."},
                    },
                    "required": ["action", "rationale"],
                },
            },
            "required": ["sections", "recommended_action"],
        },
    },
    {
        "name": "submit_for_review",
        "description": "Route the narrative and recommended action to the human reviewer and pause the pipeline. Do not call any further tools until the review response arrives. This is a mandatory gate: the agent proposes, the human disposes.",
        "input_schema": {
            "type": "object",
            "properties": {
                "message": {"type": "string", "description": "Brief message to the reviewer: what was found, what you recommend, where you are uncertain."},
            },
            "required": ["message"],
        },
    },
    {
        "name": "log_output",
        "description": "Write the complete audit trail after the reviewer's disposition: final narrative sections with reviewer actions, the disposition of the recommended action, and a one-paragraph summary.",
        "input_schema": {
            "type": "object",
            "properties": {
                "final_sections": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "final_content": {"type": "string"},
                            "reviewer_action": {"type": "string", "enum": ["approved", "edited", "rejected"]},
                            "reviewer_notes": {"type": "string"},
                        },
                        "required": ["title", "final_content", "reviewer_action"],
                    },
                },
                "recommended_action_disposition": {
                    "type": "object",
                    "properties": {
                        "action": {"type": "string"},
                        "disposition": {"type": "string", "enum": ["approved", "edited", "rejected"]},
                        "notes": {"type": "string"},
                    },
                    "required": ["action", "disposition"],
                },
                "summary": {"type": "string", "description": "One-paragraph summary of the forecast, the risk, and the review outcome."},
            },
            "required": ["final_sections", "recommended_action_disposition", "summary"],
        },
    },
]


## 1.8 · `agent.py` — the loop and the gate

The standard Anthropic tool-use loop: call the model, execute each requested tool, feed
results back, repeat until it stops. Two things make it a *harness* rather than a script:

1. **The human gate is a callback.** When the agent calls `submit_for_review`, the loop
   emits a `review_requested` event and calls whatever `review_handler` you passed in.
   This notebook passes an in-cell prompt (approve / edit / reject per section). Phase 2
   passes a WebSocket handler that blocks until the frontend responds. No handler means
   auto-approve (CLI smoke tests). Same loop, three gates.
2. **Everything is an event.** `step_started`, `step_completed`, `agent_text`,
   `review_requested`, `pipeline_complete`, `report_generated` — printed here, streamed
   over a WebSocket in Phase 2, and logged to the audit trail either way.

After the loop completes, the harness writes the sponsor deliverable (the 13-week
workbook) automatically — inside a try/except, because a report bug must never kill a
pipeline run.

In [ ]:
%%writefile cashflow_harness/agent.py
"""Agent orchestration loop: the Anthropic tool-use pattern.

The human review gate is a callback. Phase 1 (the notebook) passes a handler
that prompts in a cell; Phase 2 (the server) passes one that emits
review_requested over the WebSocket and blocks on the frontend's response.
With no handler, every section auto-approves (CLI mode).

Event protocol (kept verbatim from the HFMA build so the Phase 2 frontend
can be cloned): step_started, step_completed, agent_text, review_requested,
pipeline_complete. The frontend sends review_submitted.
"""

import asyncio
import json
from typing import Any, Awaitable, Callable, Optional

import anthropic

from .config import ANTHROPIC_API_KEY, INITIAL_PROMPT, MAX_TOKENS, MODEL, SYSTEM_PROMPT
from .logger import AuditLogger
from .tool_schemas import TOOLS
from .tools import execute_tool

WsCallback = Callable[[str, dict], Awaitable[None]]
ReviewHandler = Callable[[dict], Awaitable[dict]]

# Demo pacing: seconds to pause before each tool executes, scaled by the
# `pace` argument (0 in notebook/CLI, 1.0 for the recorded demo UI).
DEMO_DELAYS = {
    "load_facility_and_position": 3.0,
    "load_receivables": 3.5,
    "load_payables_and_fixed": 3.5,
    "build_deterministic_forecast": 4.0,
    "run_monte_carlo": 4.0,
    "run_scenario": 3.0,
    "compute_variance": 3.0,
    "draft_cash_narrative": 4.0,
    "submit_for_review": 1.0,
    "log_output": 1.5,
}


async def _noop_callback(event: str, data: dict) -> None:
    pass


async def run_agent(
    ws_callback: WsCallback = _noop_callback,
    review_handler: Optional[ReviewHandler] = None,
    audit_logger: Optional[AuditLogger] = None,
    pace: float = 0.0,
) -> dict:
    # Fall back to the live environment: notebooks often set the key after
    # this module was first imported.
    import os
    client = anthropic.AsyncAnthropic(api_key=ANTHROPIC_API_KEY or os.getenv("ANTHROPIC_API_KEY"))
    logger = audit_logger or AuditLogger()
    messages: list[dict] = [{"role": "user", "content": INITIAL_PROMPT}]
    pipeline_state: dict[str, Any] = {}

    logger.log("pipeline_started", {"prompt": INITIAL_PROMPT})

    while True:
        response = await client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        tool_results = []
        for block in response.content:
            if block.type == "text":
                await ws_callback("agent_text", {"text": block.text})
                logger.log("agent_text", {"text": block.text})

            elif block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input

                delay = DEMO_DELAYS.get(tool_name, 0) * pace
                if delay > 0:
                    await asyncio.sleep(delay)

                await ws_callback("step_started", {"step": tool_name, "input": tool_input})
                logger.log("step_started", {"step": tool_name, "input": tool_input})

                if tool_name == "submit_for_review":
                    result = await _handle_review_gate(
                        tool_input, ws_callback, review_handler, pipeline_state, logger
                    )
                else:
                    result = execute_tool(tool_name, tool_input)

                _capture_state(pipeline_state, tool_name, tool_input, result)

                await ws_callback("step_completed", {"step": tool_name, "output": result})
                logger.log("step_completed", {"step": tool_name, "output": result})

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })

        messages.append({"role": "assistant", "content": _serialize_content(response.content)})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})

        if response.stop_reason == "end_turn":
            logger.log("pipeline_complete", {})
            await ws_callback("pipeline_complete", {})
            break

    # The sponsor deliverable: a traditional 13-week cash flow workbook built
    # from the same engine the agent used. Never let a report failure break
    # the pipeline run.
    try:
        from .report import write_xlsx
        workbook_path = write_xlsx()
        pipeline_state["workbook_path"] = workbook_path
        logger.log("report_generated", {"path": workbook_path})
        await ws_callback("report_generated", {"path": workbook_path})
    except Exception as exc:  # noqa: BLE001
        logger.log("report_failed", {"error": str(exc)})

    logger.write()
    logger.write_replay()
    return {"status": "complete", "state": pipeline_state}


def _capture_state(state: dict, tool_name: str, tool_input: dict, result: dict) -> None:
    """Keep the pieces later steps and the Phase 2 UI need."""
    if tool_name == "build_deterministic_forecast":
        state["deterministic"] = result
    elif tool_name == "run_monte_carlo":
        state["monte_carlo"] = result
    elif tool_name == "run_scenario":
        state.setdefault("scenarios", []).append({"input": tool_input, "output": result})
    elif tool_name == "compute_variance":
        state["variance"] = result
    elif tool_name == "draft_cash_narrative" and "error" not in result:
        state["narrative_sections"] = tool_input.get("sections", [])
        state["recommended_action"] = tool_input.get("recommended_action", {})
    elif tool_name == "log_output":
        state["final_sections"] = tool_input.get("final_sections", [])
        state["recommended_action_disposition"] = tool_input.get("recommended_action_disposition", {})
        state["report_summary"] = tool_input.get("summary", "")


async def _handle_review_gate(
    tool_input: dict,
    ws_callback: WsCallback,
    review_handler: Optional[ReviewHandler],
    pipeline_state: dict,
    logger: AuditLogger,
) -> dict:
    payload = {
        "sections": pipeline_state.get("narrative_sections", []),
        "recommended_action": pipeline_state.get("recommended_action", {}),
        "message": tool_input.get("message", ""),
    }
    await ws_callback("review_requested", payload)
    logger.log("review_requested", payload)

    if review_handler is not None:
        review = await review_handler(payload)
    else:
        review = {
            "decisions": [
                {"section_title": s.get("title", ""), "action": "approved",
                 "edited_content": None, "notes": "Auto-approved (no review handler)."}
                for s in payload["sections"]
            ],
            "recommended_action_disposition": {
                "action": payload["recommended_action"].get("action", ""),
                "disposition": "approved",
                "notes": "Auto-approved (no review handler).",
            },
        }

    pipeline_state["review"] = review
    logger.log("review_completed", review)
    return {"status": "review_complete", **review}


def _serialize_content(content) -> list[dict]:
    serialized = []
    for block in content:
        if block.type == "text":
            serialized.append({"type": "text", "text": block.text})
        elif block.type == "tool_use":
            serialized.append({
                "type": "tool_use",
                "id": block.id,
                "name": block.name,
                "input": block.input,
            })
    return serialized


async def _cli_callback(event: str, data: dict) -> None:
    if event == "agent_text":
        print(f"\n[Agent] {data['text']}")
    elif event == "step_started":
        print(f"\n>>> {data['step']}  input={json.dumps(data['input'])[:160]}")
    elif event == "step_completed":
        out = json.dumps(data["output"])
        print(f"    done ({len(out)} bytes): {out[:220]}")
    elif event == "review_requested":
        print(f"\n{'=' * 60}\nREVIEW GATE  ({len(data.get('sections', []))} sections)")
        print(f"Recommended action: {data.get('recommended_action', {}).get('action', '')}")
        print("=" * 60)
    elif event == "pipeline_complete":
        print(f"\n{'=' * 60}\nPIPELINE COMPLETE\n{'=' * 60}")


if __name__ == "__main__":
    print("Cascade Precision Cash Flow Agent -- CLI mode (auto-approve)")
    result = asyncio.run(run_agent(ws_callback=_cli_callback))
    print(f"\nFinal status: {result['status']}")


## 1.9 · `report.py` — the sponsor deliverable

Agents that end in JSON don't get invited back. This module renders the classic FP&A
artifact — a **traditional 13-week cash flow**: line items down the side, weeks across
the top. Receipts by customer (top 5 named, the \$2.1M Vantage milestone visible in its
week-7 column), disbursements by category, cash-before-revolver mechanics, draws and
paydowns, and the revolver/covenant block ending in a PASS/BREACH flag.

`build_forecast_grid()` asserts every line ties to the engine's weekly totals **to the
dollar** — the spreadsheet cannot drift from what the agent analyzed. `write_xlsx()`
always works (openpyxl); `to_google_sheet()` publishes a real Google Sheet, which in
Colab needs nothing but one auth popup.

In [ ]:
%%writefile cashflow_harness/report.py
"""The sponsor deliverable: a traditional 13-week cash flow workbook.

Builds the classic FP&A treasury layout — line items down the side, weeks
across the top — from the same data and engine the agent used, then writes
it as a formatted Excel workbook (always) or a Google Sheet (from Colab,
after google.colab.auth). Every line ties to the engine's weekly totals to
the dollar; build_forecast_grid() asserts it.

Row model shared by both writers:
    {"label": str, "values": [13 floats] | None, "total": float | None,
     "style": "title" | "note" | "header" | "item" | "total" | "metric"
              | "flag" | "spacer"}
"""

from collections import defaultdict
from datetime import date, timedelta

from . import engine, tools
from .config import OUTPUT_DIR

TOP_CUSTOMER_ROWS = 5


def _customer_collections(invoices, week1, num_weeks):
    """AR collections bucketed by customer x week (in-horizon only)."""
    by_cust = defaultdict(lambda: [0.0] * num_weeks)
    totals = defaultdict(float)
    for inv in invoices:
        totals[inv["customer"]] += inv["amount"]
        w = engine.date_to_week(engine.expected_pay_date(inv), week1, num_weeks)
        if w <= num_weeks:
            by_cust[inv["customer"]][w - 1] += inv["amount"]
    ranked = sorted(totals, key=lambda c: -totals[c])
    return by_cust, ranked


def build_forecast_grid() -> dict:
    facility = tools._load_facility()
    invoices = tools._load_invoices()
    bills = tools._load_bills()
    fixed = tools._load_fixed()
    sales = tools._load_sales()
    week1 = tools._week1(facility)
    num_weeks = facility["num_weeks"]

    det = tools._run_deterministic(facility, invoices, bills, fixed, sales)
    weeks = det["weeks"]

    # --- Receipts: top customers, all other, forecast new billings ---------
    by_cust, ranked = _customer_collections(invoices, week1, num_weeks)
    top = ranked[:TOP_CUSTOMER_ROWS]
    other = [0.0] * num_weeks
    for c in ranked[TOP_CUSTOMER_ROWS:]:
        other = [a + b for a, b in zip(other, by_cust[c])]

    new_billings = [0.0] * num_weeks
    for row in sales:
        for amount, lag, _src in engine.split_sales_row(row):
            collect = date.fromisoformat(str(row["week_start"])) + timedelta(days=int(lag))
            w = engine.date_to_week(collect, week1, num_weeks)
            if w <= num_weeks:
                new_billings[w - 1] += amount

    receipt_rows = [
        {"label": f"{c}", "values": by_cust[c]} for c in top
    ] + [
        {"label": "All other customers", "values": other},
        {"label": "Collections on forecast new billings", "values": new_billings},
    ]
    total_receipts = [sum(r["values"][i] for r in receipt_rows) for i in range(num_weeks)]

    # --- Disbursements: trade AP + fixed schedule by category --------------
    ap_weekly = [0.0] * num_weeks
    for b in bills:
        w = engine.date_to_week(date.fromisoformat(b["due_date"]), week1, num_weeks)
        if w <= num_weeks:
            ap_weekly[w - 1] += b["amount"]

    cat_weekly = defaultdict(lambda: [0.0] * num_weeks)
    for r in fixed:
        if 1 <= r["week"] <= num_weeks:
            cat_weekly[r["category"]][r["week"] - 1] += r["amount"]

    category_labels = [
        ("materials forecast", "Material purchases (forecast ramp POs)"),
        ("payroll", "Payroll (biweekly)"),
        ("benefits", "Health benefits"),
        ("occupancy", "Rent"),
        ("operations", "Utilities & plant services"),
        ("insurance", "Insurance premium (semiannual)"),
        ("tax", "Estimated income tax"),
        ("capex", "Capex deposit (5-axis machining center)"),
        ("acquisition", "Seller note earn-out"),
        ("debt service", "Term loan debt service"),
    ]
    disb_rows = [{"label": "Trade AP (open bills)", "values": ap_weekly}] + [
        {"label": label, "values": cat_weekly[cat]}
        for cat, label in category_labels if cat in cat_weekly
    ]
    total_disb = [sum(r["values"][i] for r in disb_rows) for i in range(num_weeks)]

    # --- Tie out to the engine, to the dollar ------------------------------
    for i, w in enumerate(weeks):
        assert abs(total_receipts[i] - w["receipts"]) < 1, f"receipts mismatch wk {i + 1}"
        assert abs(total_disb[i] - w["disbursements"]) < 1, f"disbursements mismatch wk {i + 1}"

    # --- Assemble the render rows ------------------------------------------
    week_endings = [week1 + timedelta(days=4 + 7 * i) for i in range(num_weeks)]
    covenant = facility["covenant"]

    def row(label, values=None, style="item", total="sum"):
        return {
            "label": label,
            "values": list(values) if values is not None else None,
            "total": (sum(values) if (values is not None and total == "sum")
                      else (total if total != "sum" else None)),
            "style": style,
        }

    beginning = [facility["beginning_cash"]] + [w["ending_cash"] for w in weeks[:-1]]

    rows = [
        row(facility["company"], style="title"),
        row("13-Week Cash Flow Forecast — prepared for Granite Peak Partners", style="note"),
        row(f"As of the {facility['as_of_close']} close · $ whole dollars · "
            f"covenant: cash + availability ≥ ${covenant['threshold']:,.0f} at week {covenant['test_week']}",
            style="note"),
        row("", style="spacer"),
        row("CASH RECEIPTS", style="header"),
        *[row(r["label"], r["values"]) for r in receipt_rows],
        row("Total cash receipts", total_receipts, style="total"),
        row("", style="spacer"),
        row("CASH DISBURSEMENTS", style="header"),
        *[row(r["label"], r["values"]) for r in disb_rows],
        row("Total cash disbursements", total_disb, style="total"),
        row("", style="spacer"),
        row("CASH POSITION", style="header"),
        row("Beginning cash", beginning, total=None),
        row("Net cash flow", [w["net_flow"] for w in weeks], style="total"),
        row("Cash before revolver", [w["pre_revolver_cash"] for w in weeks], total=None),
        row("Revolver draw", [w["revolver_draw"] for w in weeks]),
        row("Revolver paydown", [-w["revolver_paydown"] for w in weeks]),
        row("Ending cash", [w["ending_cash"] for w in weeks], style="total", total=None),
        row(f"Memo: operating cash floor ${facility['operating_cash_floor']:,.0f}", style="note"),
        row("", style="spacer"),
        row("REVOLVER & COVENANT", style="header"),
        row("Revolver balance", [w["revolver_balance"] for w in weeks], style="metric", total=None),
        row("Undrawn availability", [w["availability"] for w in weeks], style="metric", total=None),
        row("Covenant liquidity (cash + availability)",
            [w["covenant_liquidity"] for w in weeks], style="total", total=None),
        row("Covenant threshold", [covenant["threshold"]] * num_weeks, style="metric", total=None),
        row("Headroom vs covenant",
            [w["covenant_liquidity"] - covenant["threshold"] for w in weeks],
            style="total", total=None),
        row(f"Week-{covenant['test_week']} covenant test: "
            f"{'PASS' if det['covenant_pass'] else 'BREACH'} "
            f"(headroom ${det['covenant_headroom']:,.0f})", style="flag"),
    ]

    return {
        "rows": rows,
        "week_labels": [f"Week {i + 1}" for i in range(num_weeks)],
        "week_endings": [d.isoformat() for d in week_endings],
        "num_weeks": num_weeks,
        "covenant_pass": det["covenant_pass"],
        "trough_week": det["trough_week"],
        "trough_cash": det["trough_cash"],
    }


# ---------------------------------------------------------------------------
# Excel writer
# ---------------------------------------------------------------------------

def write_xlsx(grid: dict | None = None, path=None) -> str:
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter

    grid = grid or build_forecast_grid()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = path or OUTPUT_DIR / "cashflow_13wk.xlsx"

    INK, SLATE, LIGHT, RED, GREEN = "1E293B", "3B5998", "F1F5F9", "DC2626", "16A34A"
    NUMFMT = "#,##0;(#,##0);—"
    n = grid["num_weeks"]

    wb = Workbook()
    ws = wb.active
    ws.title = "13-Week Cash Flow"
    ws.sheet_view.showGridLines = False

    header_row_ix = None
    r = 1
    for row in grid["rows"]:
        style = row["style"]
        cell = ws.cell(row=r, column=1, value=row["label"] or None)

        if style == "title":
            cell.font = Font(size=15, bold=True, color=INK)
        elif style == "note":
            cell.font = Font(size=10, italic=True, color="64748B")
        elif style == "header":
            if header_row_ix is None:
                # First section header: put the week header line right above it
                ws.insert_rows(r)
                ws.cell(row=r, column=1, value="Line item").font = Font(bold=True, color="FFFFFF")
                for i in range(n):
                    c = ws.cell(row=r, column=2 + i,
                                value=f"{grid['week_labels'][i]}\n{grid['week_endings'][i][5:]}")
                    c.font = Font(bold=True, color="FFFFFF", size=10)
                    c.alignment = Alignment(horizontal="right", wrap_text=True)
                for col in range(1, n + 3):
                    ws.cell(row=r, column=col).fill = PatternFill("solid", fgColor=SLATE)
                ws.cell(row=r, column=n + 2, value="Total").font = Font(bold=True, color="FFFFFF")
                ws.cell(row=r, column=n + 2).alignment = Alignment(horizontal="right")
                header_row_ix = r
                r += 1
                cell = ws.cell(row=r, column=1, value=row["label"])
            cell.font = Font(bold=True, color=SLATE, size=11)
            for col in range(1, n + 3):
                ws.cell(row=r, column=col).fill = PatternFill("solid", fgColor=LIGHT)
        elif style in ("item", "metric"):
            cell.font = Font(color=INK, size=11)
            cell.alignment = Alignment(indent=1)
        elif style == "total":
            cell.font = Font(bold=True, color=INK, size=11)
        elif style == "flag":
            cell.font = Font(bold=True, size=11,
                             color=(GREEN if grid["covenant_pass"] else RED))

        if row["values"] is not None:
            for i, v in enumerate(row["values"]):
                c = ws.cell(row=r, column=2 + i, value=round(v))
                c.number_format = NUMFMT
                c.font = Font(bold=(style == "total"), color=INK, size=11)
                if style == "total":
                    c.border = Border(top=Side(style="thin", color=SLATE))
            if row["total"] is not None:
                c = ws.cell(row=r, column=n + 2, value=round(row["total"]))
                c.number_format = NUMFMT
                c.font = Font(bold=True, color=INK, size=11)
        r += 1

    ws.column_dimensions["A"].width = 40
    for i in range(n + 1):
        ws.column_dimensions[get_column_letter(2 + i)].width = 12
    if header_row_ix:
        ws.freeze_panes = ws.cell(row=header_row_ix + 1, column=2)

    wb.save(path)
    return str(path)


# ---------------------------------------------------------------------------
# Google Sheets writer (Colab-friendly)
# ---------------------------------------------------------------------------

def to_google_sheet(grid: dict | None = None, title: str = "Cascade Precision — 13-Week Cash Flow") -> str:
    """Create a Google Sheet and return its URL.

    In Colab: run `from google.colab import auth; auth.authenticate_user()`
    first, then this uses your Google account directly (gspread is
    preinstalled). Locally: needs gspread plus application-default
    credentials with the Sheets scope.
    """
    try:
        import google.auth
        import gspread
    except ImportError as exc:
        raise RuntimeError(
            "gspread/google-auth not available. In Colab run "
            "`from google.colab import auth; auth.authenticate_user()` first; "
            "locally `pip install gspread google-auth`."
        ) from exc

    creds, _ = google.auth.default(scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive.file",
    ])
    gc = gspread.authorize(creds)

    grid = grid or build_forecast_grid()
    n = grid["num_weeks"]

    values, formats = [], []  # formats: (row_ix_1based, style)
    week_header = ["Line item"] + [
        f"{grid['week_labels'][i]} ({grid['week_endings'][i][5:]})" for i in range(n)
    ] + ["Total"]

    header_inserted = False
    for row in grid["rows"]:
        if row["style"] == "header" and not header_inserted:
            values.append(week_header)
            formats.append((len(values), "colhead"))
            header_inserted = True
        line = [row["label"]]
        if row["values"] is not None:
            line += [round(v) for v in row["values"]]
            line += [round(row["total"])] if row["total"] is not None else [""]
        values.append(line)
        formats.append((len(values), row["style"]))

    sh = gc.create(title)
    ws = sh.sheet1
    ws.update(values, value_input_option="RAW")

    slate, light = {"red": 0.23, "green": 0.35, "blue": 0.60}, {"red": 0.95, "green": 0.96, "blue": 0.98}
    requests = [
        {"updateSheetProperties": {"properties": {
            "sheetId": ws.id, "gridProperties": {"frozenRowCount": 0, "frozenColumnCount": 1}},
            "fields": "gridProperties.frozenColumnCount"}},
        {"repeatCell": {"range": {"sheetId": ws.id, "startColumnIndex": 1, "endColumnIndex": n + 2},
                        "cell": {"userEnteredFormat": {"numberFormat": {
                            "type": "NUMBER", "pattern": "#,##0;(#,##0);—"}}},
                        "fields": "userEnteredFormat.numberFormat"}},
        {"updateDimensionProperties": {"range": {"sheetId": ws.id, "dimension": "COLUMNS",
                                                 "startIndex": 0, "endIndex": 1},
                                       "properties": {"pixelSize": 300}, "fields": "pixelSize"}},
    ]
    for r_ix, style in formats:
        rng = {"sheetId": ws.id, "startRowIndex": r_ix - 1, "endRowIndex": r_ix}
        if style == "colhead":
            requests.append({"repeatCell": {"range": rng, "cell": {"userEnteredFormat": {
                "backgroundColor": slate,
                "textFormat": {"bold": True, "foregroundColor": {"red": 1, "green": 1, "blue": 1}}}},
                "fields": "userEnteredFormat(backgroundColor,textFormat)"}})
        elif style == "header":
            requests.append({"repeatCell": {"range": rng, "cell": {"userEnteredFormat": {
                "backgroundColor": light, "textFormat": {"bold": True}}},
                "fields": "userEnteredFormat(backgroundColor,textFormat)"}})
        elif style in ("total", "title", "flag"):
            requests.append({"repeatCell": {"range": rng, "cell": {"userEnteredFormat": {
                "textFormat": {"bold": True}}}, "fields": "userEnteredFormat.textFormat"}})
    sh.batch_update({"requests": requests})
    return sh.url


### Import the package we just wrote

In [ ]:
import pathlib
import sys

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

# __init__.py for the package
init_src = '"""Cascade Precision Products 13-week cash flow agent harness."""\n'
pathlib.Path("cashflow_harness/__init__.py").write_text(init_src)

from cashflow_harness import config
print(f"cashflow_harness ready · model = {config.MODEL} · MC = {config.MC_ITERATIONS} iterations, seed {config.MC_SEED}")

---
# Part 2 · Run it

Part 1 was the machine. Part 2 turns it on: generate the company, drive each engine by
hand so you can see what the agent will see, then hand the keys to the agent — with you
at the review gate.

## 2.1 · Verify the fixed dataset

Trust, then verify. Before any engine runs, confirm the embedded files match every
control total the demo was calibrated against: AR \$19.3M with Vantage at exactly
\$4.2M, AP \$8.7M across 90 bills, each one-timer in its scheduled week, and the
trailing 4-week miss at -6.0%. If any check fails, stop — something touched the data.

In [ ]:
import csv
import json

def check(name, ok, detail=""):
    print(f"{'PASS' if ok else 'FAIL':4}  {name}  {detail}")
    assert ok, f"Dataset verification failed: {name}"

with open("data/ar_open_invoices.json") as f:
    invoices = json.load(f)
with open("data/ap_open_bills.json") as f:
    bills = json.load(f)
with open("data/facility_terms.json") as f:
    fac = json.load(f)
with open("data/fixed_schedule.csv") as f:
    fixed = list(csv.DictReader(f))
with open("data/historical_actuals.csv") as f:
    hist = list(csv.DictReader(f))

ar_total = sum(i["amount"] for i in invoices)
vantage = sum(i["amount"] for i in invoices if i["customer"] == "Vantage Aerospace")
ap_total = sum(b["amount"] for b in bills)

check("Open AR = $19,300,000", ar_total == 19_300_000, f"${ar_total:,}")
check("122 invoices", len(invoices) == 122, str(len(invoices)))
check("Vantage = $4,200,000 (21.8%)", vantage == 4_200_000, f"${vantage:,} ({vantage/ar_total:.1%})")
check("Open AP = $8,700,000 / 90 bills", ap_total == 8_700_000 and len(bills) == 90, f"${ap_total:,} / {len(bills)}")
check("Anchor: 2026-08-21 close, week 1 = 2026-08-24",
      fac["as_of_close"] == "2026-08-21" and fac["week1_monday"] == "2026-08-24")
check("Beginning cash / revolver / covenant",
      fac["beginning_cash"] == 2_400_000 and fac["revolver_drawn"] == 6_000_000
      and fac["covenant"]["threshold"] == 4_000_000)

one_timers = {r["category"]: int(r["week"]) for r in fixed
              if r["category"] in ("insurance", "capex", "acquisition", "tax")}
check("One-timers in weeks 5/6/9/10",
      one_timers == {"insurance": 5, "capex": 6, "acquisition": 9, "tax": 10}, str(one_timers))
ds = [r for r in fixed if r["category"] == "debt service"]
check("Debt service $2.2M in week 13",
      sum(float(r["amount"]) for r in ds) == 2_200_000 and all(int(r["week"]) == 13 for r in ds))

fc = sum(float(r["forecast_collections"]) for r in hist)
ac = sum(float(r["actual_collections"]) for r in hist)
check("Trailing 4-week miss = -6.0%", abs((ac - fc) / fc + 0.06) < 0.001, f"{(ac - fc) / fc:.1%}")

print("\nDataset verified — this is the exact book the demo was calibrated on.")

## 2.2 · The position

Tool 1: the facility. Beginning cash of \$2.4M, a \$15M revolver with \$6M drawn
(\$9M availability, \$11.4M beginning liquidity), the \$1.5M floor, the \$4.0M
week-13 covenant, and \$2.2M of quarterly debt service due the same week the covenant
is tested. This is the governance frame every later number lives inside.

In [ ]:
import json
from cashflow_harness import tools

facility = tools.load_facility_and_position()
print(json.dumps(facility, indent=2))

## 2.3 · Receivables and payables — the raw material

Two things to watch in the receivables: **concentration** (top 5 ≈ 60% of AR) and the
**term-stretch flags** — customers whose observed behavior has drifted well past stated
terms. Vantage is the one that matters: 22% of AR, paying 19 days past terms. The
payables side shows the ramp: AP due dates heaviest in weeks 3–7, forecast material
purchases building through the quarter, payroll stepping up when the second shift loads
in week 5, and the five one-timers.

In [ ]:
import pandas as pd

ar = tools.load_receivables()
print(f"Open AR ${ar['total_open_ar']:,} across {ar['invoice_count']} invoices | top-5 concentration {ar['top5_concentration']:.0%}")
print(f"Term-stretch flags: {', '.join(ar['flagged_term_stretch'])}\n")

cust = pd.DataFrame(ar["customers"])
cust["open_ar"] = cust["open_ar"].map("{:,.0f}".format)
cust[["customer", "terms", "days_to_pay", "stretch_days_vs_terms", "open_ar", "share_of_ar", "term_stretch_flag"]]

In [ ]:
ap = tools.load_payables_and_fixed()
print(f"Open AP ${ap['total_open_ap']:,} across {ap['ap_bill_count']} bills\n")
print("One-time scheduled outflows:")
for item in ap["one_time_items"]:
    print(f"  wk {item['week']:>2}  ${item['amount']:>12,.0f}  {item['item']}")
print("\nDiscretionary (deferrable) items:")
for item in ap["discretionary_items"]:
    print(f"  wk {item['week']:>2}  ${item['amount']:>12,.0f}  {item['item']}")
pd.DataFrame(ap["weekly_disbursement_schedule"]).fillna(0).astype(int)

## 2.4 · The deterministic forecast

Read the story in the table: the glide down through week 4 (the \$2.1M Vantage
collection that *isn't there*), draws beginning week 5 as the insurance premium and ramp
buys hit, the **trough of ~\$1.56M in week 8** right after the \$250K week-7 draw, the
earn-out draw in week 9, and the \$2.25M debt-service draw in week 13 that leaves
**\$631,300 of covenant headroom**. Passes. Thin, but passes — hold that thought for 2.5.

In [ ]:
det = tools.build_deterministic_forecast()

def acct(x):
    return f"({abs(x):,.0f})" if x < 0 else f"{x:,.0f}"

table = pd.DataFrame(det["weeks"])
display_cols = ["week", "receipts", "disbursements", "net_flow", "pre_revolver_cash",
                "revolver_draw", "revolver_paydown", "revolver_balance", "ending_cash", "covenant_liquidity"]
styled = table[display_cols].copy()
for c in display_cols[1:]:
    styled[c] = styled[c].map(acct)
print(f"Trough: ${det['trough_cash']:,.0f} in week {det['trough_week']}")
print(f"Week-13 covenant liquidity: ${det['test_week_liquidity']:,.0f} vs ${det['covenant_threshold']:,.0f} threshold")
print(f"Covenant headroom: ${det['covenant_headroom']:,.0f}  →  {'PASS' if det['covenant_pass'] else 'BREACH'}")
styled

## 2.5 · The Monte Carlo — where the demo turns

Same engine, 1,000 seeded iterations over collection timing and sales volume. The
deterministic line said *pass, \$631K to spare*. The distribution says **the P10 path
breaches at week 13** and puts breach probability around **20%**. Nothing about the data
changed. The left chart shows weekly ending cash against the \$1.5M floor; the right
shows covenant liquidity against the \$4.0M threshold — the fan crossing that red line
is the picture that changes the sponsor conversation.

If the distribution idea still feels exotic, translate the output into the language any
treasurer already uses: *"we pass the covenant in the base case, but if Vantage and one
or two others drift the way they've been drifting, roughly one quarter in five ends in
an event of default."* That sentence was always true. The classic TWCF just couldn't
say it with a number attached — and the number is what lets you price the fix (section
2.6: accelerating Vantage to terms takes the odds from 20% to ~1%).

In [ ]:
mc = tools.run_monte_carlo()

print(f"Iterations: {mc['iterations']}  (seed {mc['seed']}, reproducible)")
print(f"P(covenant breach at week 13):        {mc['p_covenant_breach']:.1%}")
print(f"P(floor unrestorable within revolver): {mc['p_below_floor_any_week']:.1%}")
print(f"Week-13 liquidity  P10 ${mc['test_week_liquidity_p10']:,}   P50 ${mc['test_week_liquidity_p50']:,}   P90 ${mc['test_week_liquidity_p90']:,}")
print(f"Trough cash        P10 ${mc['trough_cash_p10']:,}   P50 ${mc['trough_cash_p50']:,}   P90 ${mc['trough_cash_p90']:,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

SLATE, AMBER, GREEN, RED, INK = "#3B5998", "#D97706", "#16A34A", "#DC2626", "#1E293B"
weeks = mc["weeks"]
det_cash = [w["ending_cash"] for w in det["weeks"]]
det_liq = [w["covenant_liquidity"] for w in det["weeks"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor="white")

# Left: ending cash fan with the operating floor
ax1.fill_between(weeks, mc["ending_cash_p10"], mc["ending_cash_p90"],
                 color=SLATE, alpha=0.18, label="P10–P90 band")
ax1.plot(weeks, mc["ending_cash_p50"], color=SLATE, lw=2.4, label="P50 (median)")
ax1.plot(weeks, det_cash, color=INK, lw=2, ls="--", label="Deterministic")
ax1.axhline(mc["operating_floor"], color=AMBER, lw=2, ls=":")
ax1.annotate("Operating floor $1.5M", (1, mc["operating_floor"]), xytext=(0, 8),
             textcoords="offset points", color=AMBER, fontweight="bold")
ax1.set_title("Weekly ending cash — deterministic vs. P10–P90 fan", color=INK, fontsize=13, fontweight="bold")
ax1.set_xlabel("Week"); ax1.set_xticks(weeks)

# Right: covenant liquidity fan with the threshold
ax2.fill_between(weeks, mc["liquidity_p10"], mc["liquidity_p90"],
                 color=SLATE, alpha=0.18, label="P10–P90 band")
ax2.plot(weeks, mc["liquidity_p50"], color=SLATE, lw=2.4, label="P50 (median)")
ax2.plot(weeks, det_liq, color=INK, lw=2, ls="--", label="Deterministic")
ax2.axhline(mc["covenant_threshold"], color=RED, lw=2, ls=":")
ax2.annotate("Covenant threshold $4.0M", (1, mc["covenant_threshold"]), xytext=(0, 8),
             textcoords="offset points", color=RED, fontweight="bold")
ax2.annotate(f"P(breach wk 13) = {mc['p_covenant_breach']:.0%}",
             (13, mc["liquidity_p10"][-1]), xytext=(-130, -34), textcoords="offset points",
             color=RED, fontsize=13, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.4", fc="#FEF2F2", ec=RED))
ax2.set_title("Covenant liquidity (cash + availability)", color=INK, fontsize=13, fontweight="bold")
ax2.set_xlabel("Week"); ax2.set_xticks(weeks)

for ax in (ax1, ax2):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"${v/1e6:.1f}M"))
    ax.legend(loc="upper right", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#F1F5F9")
plt.tight_layout()
plt.show()

## 2.6 · Scenarios — what actually moves the covenant

The driver sweep, ranked by impact on week-13 headroom. The finance lesson falls out of
the mechanics: covenant liquidity is invariant to revolver activity, so **only flows that
cross the week-13 boundary move deterministic headroom**. Deferring the capex deposit two
weeks (6 → 8) does *nothing* — the revolver was already defending the floor; the deferral
just changes which week draws. Deferring it *past the test date* adds \$800K. And
accelerating Vantage back to its Net 45 terms is the biggest lever on the board: +\$1.1M
of headroom, breach probability from 20% to ~1%.

In [ ]:
sweep = tools.run_scenario(scenario_type="driver_sweep")
b = sweep["baseline"]
print(f"Baseline: trough ${b['trough_cash']:,.0f} wk {b['trough_week']} | headroom ${b['covenant_headroom']:,.0f} | P(breach) {b['p_covenant_breach']:.1%}\n")
rows = pd.DataFrame(sweep["sweep"])[["label", "delta_covenant_headroom", "delta_trough_cash", "delta_p_breach", "p_covenant_breach"]]
rows

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8), facecolor="white")
data = sorted(sweep["sweep"], key=lambda r: r["delta_covenant_headroom"])
labels = [r["label"] for r in data]
vals = [r["delta_covenant_headroom"] for r in data]
colors = [GREEN if v > 0 else (RED if v < 0 else "#94A3B8") for v in vals]
ax.barh(labels, vals, color=colors)
for i, (v, r) in enumerate(zip(vals, data)):
    ax.text(v + (40_000 if v >= 0 else -40_000), i, f"P(breach) {r['p_covenant_breach']:.0%}",
            va="center", ha="left" if v >= 0 else "right", fontsize=10, color=INK)
ax.axvline(0, color=INK, lw=1)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"${v/1e6:+.1f}M"))
ax.set_title("Driver tornado — change to week-13 covenant headroom", color=INK, fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 2.7 · Trailing variance — why we trust behavior-based timing

The bridge for the last 4 weeks: collections came in 6% under the prior forecast, and
**87% of the miss is one slipped Vantage invoice** — timing on a fully collectible
receivable, not volume, not credit. That is the evidence for forecasting Vantage at 64
days instead of Net 45, and it is the same behavior driving the week-4 → week-7 slip on
the \$2.1M milestone. A forecast that can explain its own last miss earns the right to
be believed about the next thirteen weeks.

In [ ]:
var = tools.compute_variance()
print(f"Trailing {var['trailing_weeks']} weeks: forecast ${var['forecast_collections_total']:,} vs actual ${var['actual_collections_total']:,}")
print(f"Miss: ${var['total_variance']:,}  ({var['variance_pct_of_forecast']:.1%})")
print(f"Attribution — Vantage timing: ${var['attribution']['vantage_timing']:,} ({var['attribution']['vantage_share_of_miss']:.0%} of miss) | all other: ${var['attribution']['all_other']:,}")
pd.DataFrame(var["weekly_bridge"])

## 2.8 · The agent run — with a real human gate

Everything above was us driving the tools by hand. Now the agent drives: same functions,
Claude decides the sequence, reads the results, runs the scenarios it judges relevant,
and drafts the treasury narrative. Two governance features to watch:

1. **The review gate blocks.** When the agent calls `submit_for_review`, the pipeline
   stops and this cell prompts *you* to approve / edit / reject each narrative section
   and sign off on the recommended action. Your edits become the final record. The agent
   proposes; you dispose.
2. **Everything is logged.** Every tool call, input, output, and review decision lands
   in `output/audit_log.json`.

First, the two callbacks: the review handler (the notebook's implementation of the gate —
Phase 2 swaps in a WebSocket version, same agent loop) and a pretty-printer for the
event stream.

In [ ]:
# The notebook implementation of the human gate: prompt in-cell for each section.
# Phase 2 swaps this callback for a WebSocket gate in the demo UI. Same agent loop.

async def notebook_review_handler(payload):
    print("=" * 72)
    print("HUMAN REVIEW GATE — the pipeline is paused until you respond")
    print("=" * 72)
    if payload["message"]:
        print(f"\nAgent's message to you:\n{payload['message']}\n")

    decisions = []
    for s in payload["sections"]:
        print("-" * 72)
        print(f"SECTION: {s['title']}   [confidence: {s['confidence']}]")
        print(s["content"])
        for flag in s.get("open_flags", []):
            print(f"  ⚑ open flag: {flag}")
        while True:
            choice = input("[a]pprove / [e]dit / [r]eject > ").strip().lower()
            if choice in ("a", "e", "r"):
                break
        if choice == "a":
            decisions.append({"section_title": s["title"], "action": "approved",
                              "edited_content": None, "notes": ""})
        elif choice == "e":
            edited = input("Replacement text > ").strip()
            decisions.append({"section_title": s["title"], "action": "edited",
                              "edited_content": edited, "notes": "Edited at notebook gate."})
        else:
            reason = input("Rejection reason > ").strip()
            decisions.append({"section_title": s["title"], "action": "rejected",
                              "edited_content": None, "notes": reason})

    ra = payload["recommended_action"]
    print("-" * 72)
    print(f"RECOMMENDED ACTION: {ra.get('action', '')}")
    print(f"Rationale: {ra.get('rationale', '')}")
    if "expected_headroom_effect" in ra:
        print(f"Expected headroom effect: ${ra['expected_headroom_effect']:,.0f}")
    while True:
        choice = input("Sign off on the recommended action? [a]pprove / [r]eject > ").strip().lower()
        if choice in ("a", "r"):
            break
    disposition = {"action": ra.get("action", ""),
                   "disposition": "approved" if choice == "a" else "rejected",
                   "notes": "" if choice == "a" else input("Reason > ").strip()}
    print("=" * 72)
    return {"decisions": decisions, "recommended_action_disposition": disposition}

In [ ]:
# Pretty printer for pipeline events as they stream past.
STEP_LABELS = {
    "load_facility_and_position": "1 · Position and facility",
    "load_receivables": "2 · Receivables",
    "load_payables_and_fixed": "3 · Payables and fixed items",
    "build_deterministic_forecast": "4 · Deterministic forecast",
    "run_monte_carlo": "5 · Monte Carlo",
    "run_scenario": "6 · Scenario",
    "compute_variance": "7 · Trailing variance",
    "draft_cash_narrative": "8 · Treasury narrative",
    "submit_for_review": "9 · Human review",
    "log_output": "10 · Audit log",
}

async def notebook_callback(event, data):
    if event == "agent_text":
        print(f"\n💬 {data['text']}")
    elif event == "step_started":
        print(f"\n▶ {STEP_LABELS.get(data['step'], data['step'])}")
    elif event == "step_completed":
        out = data["output"]
        step = data["step"]
        if step == "build_deterministic_forecast":
            print(f"   trough ${out['trough_cash']:,.0f} (wk {out['trough_week']}) | headroom ${out['covenant_headroom']:,.0f}")
        elif step == "run_monte_carlo":
            print(f"   P(breach) {out['p_covenant_breach']:.1%} | wk-13 liq P10 ${out['test_week_liquidity_p10']:,}")
        elif step == "run_scenario":
            if "sweep" in out:
                top = out["sweep"][0]
                print(f"   sweep: biggest driver → {top['label']} (Δ headroom ${top['delta_covenant_headroom']:,.0f})")
            elif "result" in out:
                r = out["result"]
                print(f"   Δ headroom ${r['delta_covenant_headroom']:,.0f} | P(breach) {r['p_covenant_breach']:.1%}")
        elif step == "compute_variance":
            print(f"   miss ${out['total_variance']:,} ({out['variance_pct_of_forecast']:.1%}), Vantage {out['attribution']['vantage_share_of_miss']:.0%} of it")
        elif step == "draft_cash_narrative":
            print(f"   {out.get('section_count', '?')} sections drafted")
        elif step == "log_output":
            print(f"   audit written → {out.get('path', '')}")
        elif "error" in out:
            print(f"   ⚠ {out['error']}")
    elif event == "report_generated":
        print(f"\n📊 13-week cash flow workbook written → {data['path']}")

In [ ]:
from cashflow_harness.agent import run_agent
from cashflow_harness.logger import AuditLogger

audit = AuditLogger()
result = await run_agent(
    ws_callback=notebook_callback,
    review_handler=notebook_review_handler,
    audit_logger=audit,
)
print(f"\nPipeline status: {result['status']}")

## 2.9 · The final narrative and the audit trail

What you approved (or edited) at the gate is the record that goes to Granite Peak. The
audit log holds every tool call, every number, and every review decision — the answer to
"where did this figure come from" six weeks from now.

In [ ]:
state = result["state"]
ra = state.get("recommended_action_disposition", {})
print("FINAL TREASURY NARRATIVE — as reviewed")
print("=" * 72)
for s in state.get("final_sections", []):
    print(f"\n## {s['title']}  [{s['reviewer_action']}]")
    print(s["final_content"])
print("\n" + "=" * 72)
print(f"RECOMMENDED ACTION [{ra.get('disposition', '?')}]: {ra.get('action', '')}")
print(f"\nRun summary: {state.get('report_summary', '')}")

In [ ]:
audit_path = pathlib.Path("output") / "audit_log.json"
with open(audit_path) as f:
    trail = json.load(f)
from collections import Counter
counts = Counter(e["event"] for e in trail["events"])
print(f"Audit trail: {trail['event_count']} events → {audit_path}")
for event, n in counts.most_common():
    print(f"  {event:>20}: {n}")
print("\nEvery number in the narrative traces to a logged tool call. That is the point.")

## 2.10 · The sponsor deliverable — a traditional 13-week TWCF spreadsheet

Numbers in a JSON payload are for machines. What goes to Granite Peak is the classic FP&A
artifact: line items down the side, weeks across the top — receipts by customer,
disbursements by category, cash before revolver, draws and paydowns, ending cash, and the
covenant block with a PASS/BREACH flag. The agent run above already wrote the Excel
version automatically (the `report_generated` event); this cell rebuilds it so you can
inspect the grid, and the next one publishes it as a real Google Sheet.

In [ ]:
from cashflow_harness.report import build_forecast_grid, write_xlsx

grid = build_forecast_grid()
path = write_xlsx(grid)   # idempotent — the agent run already wrote this
print(f"Workbook: {path}")
print(f"Trough ${grid['trough_cash']:,.0f} in week {grid['trough_week']} | "
      f"week-13 covenant test: {'PASS' if grid['covenant_pass'] else 'BREACH'}\n")

# Preview the spine of the sheet
key_rows = [r for r in grid["rows"] if r["values"] is not None and
            (r["style"] == "total" or r["label"] in ("Beginning cash", "Revolver balance", "Undrawn availability"))]
pd.DataFrame([
    {"line item": r["label"], **{f"W{i+1}": f"{v:,.0f}" for i, v in enumerate(r["values"])}}
    for r in key_rows
])

In [ ]:
# Publish as a real Google Sheet. In Colab this needs one auth popup and
# nothing else (gspread is preinstalled). Outside Colab, the Excel file above
# is the deliverable — upload it to Drive or install gspread + google-auth.
try:
    from google.colab import auth  # type: ignore
    auth.authenticate_user()
    from cashflow_harness.report import to_google_sheet
    url = to_google_sheet(grid)
    print("Google Sheet created:", url)
except ImportError:
    print("Not running in Colab — skipping the Google Sheet, the .xlsx above is the deliverable.")

---
# What Phase 2 adds

Nothing to the logic. A FastAPI server wraps `run_agent` with a WebSocket review gate,
and a React frontend renders the pipeline spine, the **deterministic ↔ probabilistic
toggle** (the single most important interaction in the demo), the P10–P90 fan chart, the
revolver utilization panel, the covenant headroom gauge, the variance waterfall, the
driver tornado, and this same review workspace. The notebook and the UI import the same
`cashflow_harness` functions — they cannot drift.

*This notebook is generated from the package source by `notebook/build_notebook.py`;
the module cells above are byte-identical to the files the Phase 2 server imports.*